# DATASET VERIFICATION

In [1]:
import pandas as pd
import os

# ============================================================
# STEP 1: DATASET VERIFICATION
# Inclusive Preschool Education Dataset
# ============================================================

file_path = r"C:\Users\SIRPI\Downloads\inclusive_preschool_drl_dataset-selected-columns.csv"

print("=" * 80)
print("STEP 1: INCLUSIVE PRESCHOOL DATASET VERIFICATION")
print("=" * 80)

# 1. Check file existence
if not os.path.exists(file_path):
    raise FileNotFoundError(f"Dataset not found: {file_path}")

# 2. Load dataset
df = pd.read_csv(file_path)

print("\n1. DATASET OVERVIEW")
print("-" * 80)
print(f"File name             : {os.path.basename(file_path)}")
print(f"Number of records     : {df.shape[0]}")
print(f"Number of columns     : {df.shape[1]}")
print(f"Memory usage          : {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# 3. Display column names and data types
print("\n2. COLUMN NAMES AND DATA TYPES")
print("-" * 80)

column_info = pd.DataFrame({
    "Column_Name": df.columns,
    "Data_Type": df.dtypes.astype(str).values,
    "Non_Null_Count": df.notna().sum().values,
    "Missing_Count": df.isnull().sum().values,
    "Unique_Values": df.nunique(dropna=True).values
})

print(column_info.to_string(index=False))

# 4. Dataset structure
print("\n3. DATASET INFORMATION")
print("-" * 80)
df.info()

# 5. Missing values
print("\n4. MISSING VALUE ANALYSIS")
print("-" * 80)

missing_summary = pd.DataFrame({
    "Missing_Count": df.isnull().sum(),
    "Missing_Percentage": (df.isnull().mean() * 100).round(2)
})

print(missing_summary.to_string())

total_missing = int(df.isnull().sum().sum())
print(f"\nTotal missing values: {total_missing}")

# 6. Duplicate records
print("\n5. DUPLICATE RECORD ANALYSIS")
print("-" * 80)

duplicate_rows = int(df.duplicated().sum())
print(f"Duplicate rows: {duplicate_rows}")

if "participant_id" in df.columns:
    duplicate_ids = int(df["participant_id"].duplicated().sum())
    missing_ids = int(df["participant_id"].isnull().sum())

    print(f"Duplicate participant IDs: {duplicate_ids}")
    print(f"Missing participant IDs  : {missing_ids}")
else:
    print("WARNING: participant_id column is missing.")

# 7. Numerical feature summary
print("\n6. NUMERICAL FEATURE SUMMARY")
print("-" * 80)

numeric_df = df.select_dtypes(include="number")

if not numeric_df.empty:
    print(numeric_df.describe().round(3).to_string())
else:
    print("No numerical columns found.")

# 8. Categorical feature distributions
print("\n7. CATEGORICAL FEATURE DISTRIBUTIONS")
print("-" * 80)

categorical_columns = df.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

for column in categorical_columns:
    print(f"\nColumn: {column}")
    print(df[column].value_counts(dropna=False).to_string())
    print("\nPercentage distribution:")
    print(
        (df[column].value_counts(normalize=True, dropna=False) * 100)
        .round(2)
        .astype(str)
        .add("%")
        .to_string()
    )

# 9. Expected dataset schema verification
print("\n8. EXPECTED FEATURE AND TARGET VERIFICATION")
print("-" * 80)

expected_columns = [
    "participant_id",
    "age",
    "cognitive_score",
    "behavior_score",
    "social_skill_score",
    "gender",
    "socio_economic_status",
    "parental_education",
    "teacher_student_ratio",
    "interaction_frequency",
    "activity_participation",
    "infrastructure_score",
    "learning_material_availability",
    "teacher_experience_years",
    "curriculum_flexibility",
    "inclusion_support_score",
    "special_needs_support",
    "policy_effectiveness"
]

missing_columns = [
    column for column in expected_columns
    if column not in df.columns
]

unexpected_columns = [
    column for column in df.columns
    if column not in expected_columns
]

print(f"Expected number of columns: {len(expected_columns)}")
print(f"Actual number of columns  : {len(df.columns)}")
print(f"Expected columns present  : {len(expected_columns) - len(missing_columns)}")

print("\nMissing expected columns:")
if missing_columns:
    for column in missing_columns:
        print(f" - {column}")
else:
    print("None")

print("\nUnexpected columns:")
if unexpected_columns:
    for column in unexpected_columns:
        print(f" - {column}")
else:
    print("None")

if "policy_effectiveness" in df.columns:
    print("\nTarget column: policy_effectiveness")
    print("\nTarget class counts:")
    print(df["policy_effectiveness"].value_counts(dropna=False).to_string())

    print("\nTarget class percentages:")
    print(
        (df["policy_effectiveness"].value_counts(
            normalize=True, dropna=False
        ) * 100).round(2).to_string()
    )
else:
    print("\nWARNING: Target column 'policy_effectiveness' is absent.")
    print("Classification cannot be reproduced using this file alone.")

# 10. Final verification report
print("\n9. FINAL VERIFICATION REPORT")
print("=" * 80)

print(f"Dataset records                 : {len(df)}")
print(f"Dataset columns                 : {len(df.columns)}")
print(f"Total missing values            : {total_missing}")
print(f"Duplicate rows                  : {duplicate_rows}")
print(f"Expected columns missing        : {len(missing_columns)}")
print(f"Target column available         : {'Yes' if 'policy_effectiveness' in df.columns else 'No'}")

if len(df) == 8000 and len(df.columns) == 18 and not missing_columns:
    print("\nSTATUS: Expected dataset structure verified.")
else:
    print("\nSTATUS: Dataset structure does not match the expected schema.")
    print("Check the missing columns and obtain the complete dataset if needed.")

print("=" * 80)
print("STEP 1 COMPLETED")
print("=" * 80)

STEP 1: INCLUSIVE PRESCHOOL DATASET VERIFICATION

1. DATASET OVERVIEW
--------------------------------------------------------------------------------
File name             : inclusive_preschool_drl_dataset-selected-columns.csv
Number of records     : 8000
Number of columns     : 10
Memory usage          : 1.68 MB

2. COLUMN NAMES AND DATA TYPES
--------------------------------------------------------------------------------
          Column_Name Data_Type  Non_Null_Count  Missing_Count  Unique_Values
       participant_id     int64            8000              0           8000
                  age     int64            8000              0              3
      cognitive_score   float64            8000              0           7981
       behavior_score   float64            8000              0           7984
   social_skill_score   float64            8000              0           7991
               gender    object            8000              0              2
socio_economic_status    

In [2]:
import os
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# ============================================================
# STEP 2: DATA PREPARATION AND VALIDATION
# Inclusive Preschool Education Dataset
# ============================================================

input_path = r"C:\Users\SIRPI\Downloads\inclusive_preschool_drl_dataset-selected-columns.csv"

output_dir = r"C:\Users\SIRPI\Downloads\Inclusive_Preschool_Step2"
os.makedirs(output_dir, exist_ok=True)

prepared_path = os.path.join(
    output_dir, "inclusive_preschool_prepared_features.csv"
)

encoded_path = os.path.join(
    output_dir, "inclusive_preschool_encoded_scaled_features.csv"
)

# ------------------------------------------------------------
# 1. LOAD DATASET
# ------------------------------------------------------------
print("=" * 80)
print("STEP 2: DATA PREPARATION AND VALIDATION")
print("=" * 80)

df = pd.read_csv(input_path)

print("\n1. ORIGINAL DATASET")
print("-" * 80)
print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())

# ------------------------------------------------------------
# 2. CHECK REQUIRED DATA
# ------------------------------------------------------------
print("\n2. DATA VALIDATION")
print("-" * 80)

print("Missing values:", df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())

if "participant_id" in df.columns:
    print("Duplicate participant IDs:",
          df["participant_id"].duplicated().sum())

# Check the expected target
target_column = "policy_effectiveness"

if target_column not in df.columns:
    print("\nWARNING: Target column is not available.")
    print("Only feature preparation will be performed.")
    print("Classification and supervised model training cannot")
    print("be performed until the genuine target labels are obtained.")

# ------------------------------------------------------------
# 3. REMOVE DUPLICATE ROWS IF ANY
# ------------------------------------------------------------
df = df.drop_duplicates().copy()

print("\n3. DUPLICATE HANDLING")
print("-" * 80)
print("Shape after duplicate check:", df.shape)

# ------------------------------------------------------------
# 4. SEPARATE IDENTIFIER FROM FEATURES
# ------------------------------------------------------------
id_column = "participant_id"

if id_column in df.columns:
    participant_ids = df[id_column].copy()
    X = df.drop(columns=[id_column]).copy()
else:
    participant_ids = pd.Series(
        np.arange(1, len(df) + 1),
        name="participant_id"
    )
    X = df.copy()

print("\n4. FEATURE SEPARATION")
print("-" * 80)
print("Identifier excluded from model inputs:", id_column)
print("Feature matrix shape:", X.shape)

# Do not accidentally include the target as an input feature.
if target_column in X.columns:
    X = X.drop(columns=[target_column])

# ------------------------------------------------------------
# 5. IDENTIFY NUMERICAL AND CATEGORICAL FEATURES
# ------------------------------------------------------------
numeric_columns = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_columns = X.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

print("\n5. FEATURE TYPES")
print("-" * 80)
print("Numerical features:", numeric_columns)
print("Number of numerical features:", len(numeric_columns))
print("\nCategorical features:", categorical_columns)
print("Number of categorical features:", len(categorical_columns))

# ------------------------------------------------------------
# 6. PREPARE NUMERICAL AND CATEGORICAL FEATURES
# ------------------------------------------------------------
numeric_pipeline = Pipeline(
    steps=[
        ("scaler", StandardScaler())
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("encoder", OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        ))
    ]
)

transformers = []

if numeric_columns:
    transformers.append(
        ("numeric", numeric_pipeline, numeric_columns)
    )

if categorical_columns:
    transformers.append(
        ("categorical", categorical_pipeline, categorical_columns)
    )

preprocessor = ColumnTransformer(
    transformers=transformers,
    remainder="drop",
    verbose_feature_names_out=False
)

# Fit and transform the available features.
X_transformed = preprocessor.fit_transform(X)

feature_names = preprocessor.get_feature_names_out()

X_prepared = pd.DataFrame(
    X_transformed,
    columns=feature_names,
    index=X.index
)

# ------------------------------------------------------------
# 7. VALIDATE PREPARED FEATURES
# ------------------------------------------------------------
print("\n6. PREPARED FEATURE VALIDATION")
print("-" * 80)

print("Prepared feature shape:", X_prepared.shape)
print("Missing values after transformation:",
      X_prepared.isnull().sum().sum())

print("\nPrepared feature sample:")
print(X_prepared.head().round(4).to_string(index=False))

print("\nPrepared feature summary:")
print(X_prepared.describe().round(4).to_string())

# ------------------------------------------------------------
# 8. SAVE PREPARED DATASETS
# ------------------------------------------------------------
# Save original available columns after duplicate removal.
df.to_csv(prepared_path, index=False)

# Save transformed features with the identifier retained
# only for record tracking, not as a model input.
output_df = pd.DataFrame({
    "participant_id": participant_ids.to_numpy()
})

output_df = pd.concat(
    [
        output_df.reset_index(drop=True),
        X_prepared.reset_index(drop=True)
    ],
    axis=1
)

output_df.to_csv(encoded_path, index=False)

print("\n7. OUTPUT FILES")
print("-" * 80)
print("Validated dataset:", prepared_path)
print("Encoded/scaled features:", encoded_path)

# ------------------------------------------------------------
# 9. FINAL REPORT
# ------------------------------------------------------------
print("\n8. FINAL STEP 2 REPORT")
print("=" * 80)
print("Records prepared:", len(X_prepared))
print("Original feature columns used:", X.shape[1])
print("Numerical input features:", len(numeric_columns))
print("Categorical input features:", len(categorical_columns))
print("Encoded feature count:", X_prepared.shape[1])
print("Missing values after preparation:",
      int(X_prepared.isnull().sum().sum()))
print("Target available:", target_column in df.columns)

if target_column not in df.columns:
    print("\nSTATUS: Feature preparation completed.")
    print("NEXT REQUIREMENT: Obtain the complete dataset with")
    print("the genuine policy_effectiveness target.")
else:
    print("\nSTATUS: Feature preparation completed.")
    
print("=" * 80)
print("STEP 2 COMPLETED")
print("=" * 80)

STEP 2: DATA PREPARATION AND VALIDATION

1. ORIGINAL DATASET
--------------------------------------------------------------------------------
Dataset shape: (8000, 10)
Columns: ['participant_id', 'age', 'cognitive_score', 'behavior_score', 'social_skill_score', 'gender', 'socio_economic_status', 'parental_education', 'teacher_student_ratio', 'interaction_frequency']

2. DATA VALIDATION
--------------------------------------------------------------------------------
Missing values: 0
Duplicate rows: 0
Duplicate participant IDs: 0

Only feature preparation will be performed.
Classification and supervised model training cannot
be performed until the genuine target labels are obtained.

3. DUPLICATE HANDLING
--------------------------------------------------------------------------------
Shape after duplicate check: (8000, 10)

4. FEATURE SEPARATION
--------------------------------------------------------------------------------
Identifier excluded from model inputs: participant_id
Feature

In [3]:
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# ============================================================
# STEP 3: TARGET AND STATE DEFINITION
# Inclusive Preschool Education Dataset
# ============================================================

input_path = (
    r"C:\Users\SIRPI\Downloads"
    r"\inclusive_preschool_drl_dataset-selected-columns.csv"
)

output_dir = r"C:\Users\SIRPI\Downloads\Inclusive_Preschool_Step3"
os.makedirs(output_dir, exist_ok=True)

state_output_path = os.path.join(
    output_dir, "inclusive_preschool_state_matrix.csv"
)

state_definition_path = os.path.join(
    output_dir, "inclusive_preschool_state_definition.csv"
)

print("=" * 80)
print("STEP 3: TARGET AND STATE DEFINITION")
print("=" * 80)

# ------------------------------------------------------------
# 1. LOAD DATASET
# ------------------------------------------------------------
df = pd.read_csv(input_path)

print("\n1. DATASET INFORMATION")
print("-" * 80)
print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())

# ------------------------------------------------------------
# 2. VERIFY TARGET COLUMN
# ------------------------------------------------------------
target_column = "policy_effectiveness"

print("\n2. TARGET VERIFICATION")
print("-" * 80)

if target_column in df.columns:
    print("Target column found:", target_column)
    print("\nTarget class counts:")
    print(df[target_column].value_counts(dropna=False))

    print("\nTarget missing values:",
          df[target_column].isnull().sum())

    print("\nTarget class percentages:")
    print(
        (df[target_column].value_counts(normalize=True) * 100)
        .round(2)
    )
else:
    print("WARNING: Target column is missing.")
    print("Expected target:", target_column)
    print("Classification target cannot be defined from this file.")
    print("No target labels will be created or inferred.")

# ------------------------------------------------------------
# 3. DEFINE AVAILABLE STATE VARIABLES
# ------------------------------------------------------------
# Exclude the participant identifier because it is not an
# educational state measurement.

state_columns = [
    "age",
    "cognitive_score",
    "behavior_score",
    "social_skill_score",
    "gender",
    "socio_economic_status",
    "parental_education",
    "teacher_student_ratio",
    "interaction_frequency"
]

available_state_columns = [
    col for col in state_columns if col in df.columns
]

missing_state_columns = [
    col for col in state_columns if col not in df.columns
]

print("\n3. STATE VARIABLE VERIFICATION")
print("-" * 80)
print("Available state variables:")
for col in available_state_columns:
    print(" -", col)

print("\nMissing expected state variables:")
if missing_state_columns:
    for col in missing_state_columns:
        print(" -", col)
else:
    print("None")

# ------------------------------------------------------------
# 4. BUILD THE STATE DATAFRAME
# ------------------------------------------------------------
state_df = df[available_state_columns].copy()

# Separate numerical and categorical state variables.
numeric_columns = state_df.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_columns = state_df.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

print("\n4. STATE VARIABLE TYPES")
print("-" * 80)
print("Numerical state variables:", numeric_columns)
print("Categorical state variables:", categorical_columns)

# ------------------------------------------------------------
# 5. ENCODE CATEGORICAL VARIABLES AND SCALE NUMERICAL ONES
# ------------------------------------------------------------
transformers = []

if numeric_columns:
    transformers.append(
        (
            "numeric",
            StandardScaler(),
            numeric_columns
        )
    )

if categorical_columns:
    transformers.append(
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_columns
        )
    )

preprocessor = ColumnTransformer(
    transformers=transformers,
    remainder="drop",
    verbose_feature_names_out=False
)

state_values = preprocessor.fit_transform(state_df)
state_feature_names = preprocessor.get_feature_names_out()

state_matrix = pd.DataFrame(
    state_values,
    columns=state_feature_names,
    index=df.index
)

# Retain IDs for tracing records, but do not use IDs as state features.
if "participant_id" in df.columns:
    state_matrix.insert(
        0,
        "participant_id",
        df["participant_id"].values
    )

print("\n5. STATE MATRIX")
print("-" * 80)
print("State matrix shape including ID:", state_matrix.shape)
print("Number of state features:", len(state_feature_names))
print("Missing values:", state_matrix.isnull().sum().sum())

print("\nState feature names:")
for feature in state_feature_names:
    print(" -", feature)

print("\nFirst five state records:")
print(state_matrix.head().round(4).to_string(index=False))

# ------------------------------------------------------------
# 6. CREATE STATE DEFINITION TABLE
# ------------------------------------------------------------
definition_rows = []

for col in available_state_columns:
    if col in numeric_columns:
        definition_rows.append({
            "State_Variable": col,
            "Variable_Type": "Numerical",
            "Description": "Available numerical participant/classroom measurement",
            "Encoding": "StandardScaler"
        })
    else:
        definition_rows.append({
            "State_Variable": col,
            "Variable_Type": "Categorical",
            "Description": "Available categorical participant/classroom attribute",
            "Encoding": "One-hot encoding"
        })

definition_df = pd.DataFrame(definition_rows)

print("\n6. STATE DEFINITION TABLE")
print("-" * 80)
print(definition_df.to_string(index=False))

# ------------------------------------------------------------
# 7. SAVE OUTPUTS
# ------------------------------------------------------------
state_matrix.to_csv(state_output_path, index=False)
definition_df.to_csv(state_definition_path, index=False)

print("\n7. SAVED OUTPUT FILES")
print("-" * 80)
print("State matrix:", state_output_path)
print("State definition:", state_definition_path)

# ------------------------------------------------------------
# 8. FINAL STATUS
# ------------------------------------------------------------
print("\n8. FINAL STEP 3 REPORT")
print("=" * 80)
print("Records:", len(df))
print("Available state variables:", len(available_state_columns))
print("Encoded state features:", len(state_feature_names))
print("Target available:", target_column in df.columns)

if target_column not in df.columns:
    print("\nSTATUS: State representation prepared for exploration.")
    print("BLOCKER: Genuine policy_effectiveness labels are unavailable.")
    print("The RL action space, reward function, and transitions")
    print("must be defined from the documented methodology and data.")
else:
    print("\nSTATUS: State representation and target verified.")

print("=" * 80)
print("STEP 3 COMPLETED")
print("=" * 80)

STEP 3: TARGET AND STATE DEFINITION

1. DATASET INFORMATION
--------------------------------------------------------------------------------
Dataset shape: (8000, 10)
Columns: ['participant_id', 'age', 'cognitive_score', 'behavior_score', 'social_skill_score', 'gender', 'socio_economic_status', 'parental_education', 'teacher_student_ratio', 'interaction_frequency']

2. TARGET VERIFICATION
--------------------------------------------------------------------------------
Expected target: policy_effectiveness
Classification target cannot be defined from this file.
No target labels will be created or inferred.

3. STATE VARIABLE VERIFICATION
--------------------------------------------------------------------------------
Available state variables:
 - age
 - cognitive_score
 - behavior_score
 - social_skill_score
 - gender
 - socio_economic_status
 - parental_education
 - teacher_student_ratio
 - interaction_frequency

Missing expected state variables:
None

4. STATE VARIABLE TYPES
---------

In [4]:
import os
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline

# ============================================================
# STEP 4: TRAIN-TEST SPLIT AND LEAKAGE PREVENTION
# Inclusive Preschool Education Dataset
# ============================================================

input_path = (
    r"C:\Users\SIRPI\Downloads"
    r"\inclusive_preschool_drl_dataset-selected-columns.csv"
)

output_dir = r"C:\Users\SIRPI\Downloads\Inclusive_Preschool_Step4"
os.makedirs(output_dir, exist_ok=True)

train_path = os.path.join(
    output_dir, "inclusive_preschool_train_features.csv"
)

test_path = os.path.join(
    output_dir, "inclusive_preschool_test_features.csv"
)

train_id_path = os.path.join(
    output_dir, "inclusive_preschool_train_ids.csv"
)

test_id_path = os.path.join(
    output_dir, "inclusive_preschool_test_ids.csv"
)

# ------------------------------------------------------------
# 1. LOAD DATASET
# ------------------------------------------------------------
print("=" * 80)
print("STEP 4: TRAIN-TEST SPLIT AND LEAKAGE PREVENTION")
print("=" * 80)

df = pd.read_csv(input_path)

print("\n1. ORIGINAL DATASET")
print("-" * 80)
print("Dataset shape:", df.shape)
print("Missing values:", df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())

# Remove exact duplicate rows if present.
df = df.drop_duplicates().copy()

# ------------------------------------------------------------
# 2. SEPARATE IDENTIFIER, FEATURES, AND TARGET
# ------------------------------------------------------------
id_column = "participant_id"
target_column = "policy_effectiveness"

if id_column in df.columns:
    record_ids = df[id_column].copy()
    X = df.drop(columns=[id_column]).copy()
else:
    record_ids = pd.Series(
        np.arange(len(df)), name="record_index"
    )
    X = df.copy()

target_available = target_column in X.columns

if target_available:
    y = X[target_column].copy()
    X = X.drop(columns=[target_column])
else:
    y = None
    print("\nWARNING: policy_effectiveness is missing.")
    print("A feature-only split will be performed.")
    print("This split cannot be used for supervised evaluation.")

print("\n2. FEATURE INFORMATION")
print("-" * 80)
print("Feature count before encoding:", X.shape[1])
print("Target available:", target_available)

# ------------------------------------------------------------
# 3. SPLIT RECORDS BEFORE FITTING TRANSFORMATIONS
# ------------------------------------------------------------
indices = np.arange(len(X))

if target_available and y.notna().all() and y.nunique() > 1:
    # Stratification is used only when genuine labels exist.
    train_idx, test_idx = train_test_split(
        indices,
        test_size=0.20,
        random_state=42,
        stratify=y
    )
else:
    train_idx, test_idx = train_test_split(
        indices,
        test_size=0.20,
        random_state=42,
        shuffle=True
    )

X_train_raw = X.iloc[train_idx].copy()
X_test_raw = X.iloc[test_idx].copy()

ids_train = record_ids.iloc[train_idx].reset_index(drop=True)
ids_test = record_ids.iloc[test_idx].reset_index(drop=True)

print("\n3. TRAIN-TEST SPLIT")
print("-" * 80)
print("Training records:", len(X_train_raw))
print("Testing records :", len(X_test_raw))
print("Training percentage:",
      round(len(X_train_raw) / len(X) * 100, 2), "%")
print("Testing percentage :",
      round(len(X_test_raw) / len(X) * 100, 2), "%")

# ------------------------------------------------------------
# 4. IDENTIFY FEATURE TYPES USING TRAINING DATA
# ------------------------------------------------------------
numeric_columns = X_train_raw.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_columns = X_train_raw.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

print("\n4. FEATURE TYPES")
print("-" * 80)
print("Numerical features:", numeric_columns)
print("Categorical features:", categorical_columns)

# ------------------------------------------------------------
# 5. CREATE PREPROCESSING PIPELINE
# Fit transformations on training data only.
# ------------------------------------------------------------
transformers = []

if numeric_columns:
    numeric_pipeline = Pipeline([
        ("scaler", StandardScaler())
    ])

    transformers.append(
        ("numeric", numeric_pipeline, numeric_columns)
    )

if categorical_columns:
    categorical_pipeline = Pipeline([
        ("encoder", OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        ))
    ])

    transformers.append(
        ("categorical", categorical_pipeline, categorical_columns)
    )

preprocessor = ColumnTransformer(
    transformers=transformers,
    remainder="drop",
    verbose_feature_names_out=False
)

# Fit only on training records.
X_train_array = preprocessor.fit_transform(X_train_raw)

# Apply the training-fitted transformations to the test set.
X_test_array = preprocessor.transform(X_test_raw)

feature_names = preprocessor.get_feature_names_out()

X_train = pd.DataFrame(
    X_train_array,
    columns=feature_names
)

X_test = pd.DataFrame(
    X_test_array,
    columns=feature_names
)

print("\n5. PREPROCESSING RESULTS")
print("-" * 80)
print("Training feature matrix:", X_train.shape)
print("Testing feature matrix :", X_test.shape)
print("Training missing values:", X_train.isnull().sum().sum())
print("Testing missing values :", X_test.isnull().sum().sum())

# ------------------------------------------------------------
# 6. VERIFY NO RECORD OVERLAP
# ------------------------------------------------------------
print("\n6. LEAKAGE AND OVERLAP CHECK")
print("-" * 80)

train_id_set = set(ids_train.tolist())
test_id_set = set(ids_test.tolist())

overlap = train_id_set.intersection(test_id_set)

print("Training IDs:", len(train_id_set))
print("Testing IDs :", len(test_id_set))
print("Overlapping IDs:", len(overlap))

assert len(overlap) == 0, "Data leakage: overlapping record IDs found!"
assert len(X_train) + len(X_test) == len(X), "Split count mismatch!"
assert list(X_train.columns) == list(X_test.columns), \
    "Training and test feature columns do not match!"

print("Record overlap check: PASSED")
print("Feature consistency check: PASSED")

# ------------------------------------------------------------
# 7. SAVE FEATURES AND RECORD IDS
# ------------------------------------------------------------
X_train.to_csv(train_path, index=False)
X_test.to_csv(test_path, index=False)

ids_train.to_frame().to_csv(train_id_path, index=False)
ids_test.to_frame().to_csv(test_id_path, index=False)

print("\n7. OUTPUT FILES")
print("-" * 80)
print("Training features:", train_path)
print("Testing features :", test_path)
print("Training IDs     :", train_id_path)
print("Testing IDs      :", test_id_path)

# ------------------------------------------------------------
# 8. FINAL REPORT
# ------------------------------------------------------------
print("\n8. FINAL STEP 4 REPORT")
print("=" * 80)
print("Total usable records:", len(X))
print("Training records:", len(X_train))
print("Testing records :", len(X_test))
print("Encoded features:", X_train.shape[1])
print("Target available:", target_available)
print("Preprocessing fitted on training set only: YES")
print("Overlapping records: NO")

if target_available:
    print("STATUS: Feature split completed; target was present.")
else:
    print("STATUS: Feature-only split completed.")
    print("NEXT REQUIREMENT: Obtain genuine policy_effectiveness labels.")

print("=" * 80)
print("STEP 4 COMPLETED")
print("=" * 80)

STEP 4: TRAIN-TEST SPLIT AND LEAKAGE PREVENTION

1. ORIGINAL DATASET
--------------------------------------------------------------------------------
Dataset shape: (8000, 10)
Missing values: 0
Duplicate rows: 0

A feature-only split will be performed.
This split cannot be used for supervised evaluation.

2. FEATURE INFORMATION
--------------------------------------------------------------------------------
Feature count before encoding: 9
Target available: False

3. TRAIN-TEST SPLIT
--------------------------------------------------------------------------------
Training records: 6400
Testing records : 1600
Training percentage: 80.0 %
Testing percentage : 20.0 %

4. FEATURE TYPES
--------------------------------------------------------------------------------
Numerical features: ['age', 'cognitive_score', 'behavior_score', 'social_skill_score', 'teacher_student_ratio', 'interaction_frequency']
Categorical features: ['gender', 'socio_economic_status', 'parental_education']

5. PREPROCE

In [5]:
import os
import numpy as np
import pandas as pd

# ============================================================
# STEP 5: REINFORCEMENT LEARNING ENVIRONMENT SETUP
# Inclusive Preschool Education
# Provisional environment for technical testing only
# ============================================================

input_path = (
    r"C:\Users\SIRPI\Downloads\Inclusive_Preschool_Step4"
    r"\inclusive_preschool_train_features.csv"
)

output_dir = r"C:\Users\SIRPI\Downloads\Inclusive_Preschool_Step5"
os.makedirs(output_dir, exist_ok=True)

transition_path = os.path.join(
    output_dir, "provisional_rl_transition_log.csv"
)

print("=" * 80)
print("STEP 5: REINFORCEMENT LEARNING ENVIRONMENT SETUP")
print("=" * 80)

# ------------------------------------------------------------
# 1. LOAD TRAINING FEATURES
# ------------------------------------------------------------
df = pd.read_csv(input_path)

# The participant ID was excluded from model inputs in Step 4.
state_columns = df.columns.tolist()
states = df[state_columns].to_numpy(dtype=np.float32)

print("\n1. STATE SPACE")
print("-" * 80)
print("Training records:", len(df))
print("State dimension:", states.shape[1])
print("State variables:")
for col in state_columns:
    print(" -", col)

# ------------------------------------------------------------
# 2. DEFINE A PROVISIONAL DISCRETE ACTION SPACE
# ------------------------------------------------------------
# These are illustrative intervention codes, not validated
# policies or recommendations for actual children.
actions = {
    0: "Maintain current support",
    1: "Increase teacher interaction",
    2: "Provide additional learning materials",
    3: "Increase inclusive learning support"
}

print("\n2. PROVISIONAL ACTION SPACE")
print("-" * 80)

for action_id, action_name in actions.items():
    print(f"Action {action_id}: {action_name}")

print("\nNumber of actions:", len(actions))

# ------------------------------------------------------------
# 3. DEFINE ENVIRONMENT
# ------------------------------------------------------------
class InclusivePreschoolEnvironment:
    """
    Provisional environment for software testing.

    IMPORTANT:
    - Does not model validated real-world policy effects.
    - Uses a placeholder reward for technical demonstration.
    - Must be replaced with a manuscript-supported environment
      before drawing scientific conclusions.
    """

    def __init__(self, state_array, action_dictionary, max_steps=10):
        self.states = np.asarray(state_array, dtype=np.float32)
        self.actions = action_dictionary
        self.max_steps = max_steps

        if len(self.states) == 0:
            raise ValueError("The state dataset is empty.")

        self.current_index = 0
        self.step_count = 0
        self.current_state = None

    def reset(self, index=0):
        if index < 0 or index >= len(self.states):
            raise IndexError("Initial state index is out of range.")

        self.current_index = index
        self.step_count = 0
        self.current_state = self.states[index].copy()

        return self.current_state.copy()

    def step(self, action):
        if action not in self.actions:
            raise ValueError("Invalid action ID.")

        if self.current_state is None:
            raise RuntimeError("Call reset() before step().")

        previous_state = self.current_state.copy()

        # Placeholder transition:
        # move to the next observed record. This is NOT a
        # causal or real-world intervention transition.
        next_index = min(
            self.current_index + 1,
            len(self.states) - 1
        )

        next_state = self.states[next_index].copy()

        # Placeholder reward: zero for all actions.
        # No evidence currently supports a reward for the
        # proposed actions, so no artificial improvement is
        # assigned to any action.
        reward = 0.0

        self.current_index = next_index
        self.current_state = next_state
        self.step_count += 1

        terminated = (
            self.current_index >= len(self.states) - 1
        )

        truncated = self.step_count >= self.max_steps

        info = {
            "action_name": self.actions[action],
            "reward_is_placeholder": True,
            "transition_is_observed_row_sequence": True
        }

        return (
            next_state.copy(),
            reward,
            terminated,
            truncated,
            info
        )

# ------------------------------------------------------------
# 4. INITIALIZE ENVIRONMENT
# ------------------------------------------------------------
env = InclusivePreschoolEnvironment(
    state_array=states,
    action_dictionary=actions,
    max_steps=10
)

initial_state = env.reset(index=0)

print("\n3. ENVIRONMENT INITIALIZATION")
print("-" * 80)
print("Environment initialized successfully.")
print("Initial state dimension:", len(initial_state))
print("Maximum steps per episode:", env.max_steps)

# ------------------------------------------------------------
# 5. TEST ONE EPISODE
# ------------------------------------------------------------
print("\n4. ENVIRONMENT STEP TEST")
print("-" * 80)

state = env.reset(index=0)
transition_records = []

for step_number in range(1, 11):
    # Action 0 is selected only to test the interface.
    # This is not a learned or recommended policy.
    action = 0

    next_state, reward, terminated, truncated, info = env.step(action)

    transition_records.append({
        "Step": step_number,
        "Action_ID": action,
        "Action_Name": info["action_name"],
        "Reward": reward,
        "Next_State_Index": env.current_index,
        "Terminated": terminated,
        "Truncated": truncated,
        "Reward_Is_Placeholder": info["reward_is_placeholder"],
        "Transition_Is_Observed_Row_Sequence":
            info["transition_is_observed_row_sequence"]
    })

    print(
        f"Step {step_number:02d} | "
        f"Action: {action} | "
        f"Reward: {reward:.2f} | "
        f"Terminated: {terminated} | "
        f"Truncated: {truncated}"
    )

    state = next_state

    if terminated or truncated:
        break

# ------------------------------------------------------------
# 6. SAVE TEST TRANSITIONS
# ------------------------------------------------------------
transition_df = pd.DataFrame(transition_records)
transition_df.to_csv(transition_path, index=False)

print("\n5. TRANSITION LOG")
print("-" * 80)
print(transition_df.to_string(index=False))
print("\nSaved transition log:", transition_path)

# ------------------------------------------------------------
# 7. FINAL REPORT
# ------------------------------------------------------------
print("\n6. FINAL STEP 5 REPORT")
print("=" * 80)
print("State dimension:", states.shape[1])
print("Number of provisional actions:", len(actions))
print("Transitions tested:", len(transition_df))
print("Transition log saved: YES")
print("Policy effectiveness target available: NO")
print("Reward function validated: NO")
print("Real-world intervention dynamics validated: NO")

print("\nSTATUS: Provisional environment interface tested.")
print("REQUIRED BEFORE MODEL TRAINING:")
print("1. Recover the complete dataset and genuine target, if required.")
print("2. Define actions from the manuscript's methodology.")
print("3. Implement evidence-based state transitions.")
print("4. Implement and justify the reward function.")
print("5. Validate the environment before evaluating DE-TwinD3PG.")

print("=" * 80)
print("STEP 5 COMPLETED")
print("=" * 80)

STEP 5: REINFORCEMENT LEARNING ENVIRONMENT SETUP

1. STATE SPACE
--------------------------------------------------------------------------------
Training records: 6400
State dimension: 14
State variables:
 - age
 - cognitive_score
 - behavior_score
 - social_skill_score
 - teacher_student_ratio
 - interaction_frequency
 - gender_Female
 - gender_Male
 - socio_economic_status_High
 - socio_economic_status_Low
 - socio_economic_status_Middle
 - parental_education_Higher
 - parental_education_Primary
 - parental_education_Secondary

2. PROVISIONAL ACTION SPACE
--------------------------------------------------------------------------------
Action 0: Maintain current support
Action 1: Increase teacher interaction
Action 2: Provide additional learning materials
Action 3: Increase inclusive learning support

Number of actions: 4

3. ENVIRONMENT INITIALIZATION
--------------------------------------------------------------------------------
Environment initialized successfully.
Initial state 

In [6]:
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim

# ============================================================
# STEP 6: PROVISIONAL TD3 ACTOR-CRITIC MODEL
# Inclusive Preschool Education
#
# IMPORTANT:
# This is an architecture and execution test only.
# It is NOT a validated DE-TwinD3PG implementation.
# Do not use its outputs as research performance results.
# ============================================================

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

input_path = (
    r"C:\Users\SIRPI\Downloads\Inclusive_Preschool_Step4"
    r"\inclusive_preschool_train_features.csv"
)

output_dir = r"C:\Users\SIRPI\Downloads\Inclusive_Preschool_Step6"
os.makedirs(output_dir, exist_ok=True)

model_path = os.path.join(
    output_dir, "provisional_td3_actor_critic.pth"
)

# ------------------------------------------------------------
# 1. LOAD STATE FEATURES
# ------------------------------------------------------------
print("=" * 80)
print("STEP 6: PROVISIONAL ACTOR-CRITIC MODEL")
print("=" * 80)

df = pd.read_csv(input_path)
states_np = df.to_numpy(dtype=np.float32)

if not np.isfinite(states_np).all():
    raise ValueError("State matrix contains NaN or infinite values.")

states = torch.tensor(states_np, dtype=torch.float32, device=device)

state_dim = states.shape[1]
action_dim = 4

print("\n1. MODEL INPUT")
print("-" * 80)
print("Training records:", len(df))
print("State dimension:", state_dim)
print("Action dimension:", action_dim)
print("Device:", device)

# ------------------------------------------------------------
# 2. ACTOR NETWORK
# Maps a state to four continuous action scores.
# These scores are NOT validated intervention decisions.
# ------------------------------------------------------------
class Actor(nn.Module):
    def __init__(self, state_dim, action_dim):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(state_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, action_dim),
            nn.Tanh()
        )

    def forward(self, state):
        return self.network(state)


# ------------------------------------------------------------
# 3. CRITIC NETWORK
# Estimates Q(state, action).
# ------------------------------------------------------------
class Critic(nn.Module):
    def __init__(self, state_dim, action_dim):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(state_dim + action_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 1)
        )

    def forward(self, state, action):
        combined = torch.cat([state, action], dim=1)
        return self.network(combined)


actor = Actor(state_dim, action_dim).to(device)
actor_target = Actor(state_dim, action_dim).to(device)

critic1 = Critic(state_dim, action_dim).to(device)
critic2 = Critic(state_dim, action_dim).to(device)

critic1_target = Critic(state_dim, action_dim).to(device)
critic2_target = Critic(state_dim, action_dim).to(device)

actor_target.load_state_dict(actor.state_dict())
critic1_target.load_state_dict(critic1.state_dict())
critic2_target.load_state_dict(critic2.state_dict())

actor_optimizer = optim.Adam(actor.parameters(), lr=1e-3)
critic1_optimizer = optim.Adam(critic1.parameters(), lr=1e-3)
critic2_optimizer = optim.Adam(critic2.parameters(), lr=1e-3)

print("\n2. NETWORK ARCHITECTURE")
print("-" * 80)
print("Actor: State -> 256 -> 128 -> 4")
print("Critic 1: State + Action -> 256 -> 128 -> Q-value")
print("Critic 2: State + Action -> 256 -> 128 -> Q-value")
print("Target actor and target critics initialized.")

# ------------------------------------------------------------
# 4. TEST FORWARD PASSES
# ------------------------------------------------------------
print("\n3. FORWARD-PASS VALIDATION")
print("-" * 80)

with torch.no_grad():
    sample_states = states[:5]
    sample_actions = actor(sample_states)
    sample_q1 = critic1(sample_states, sample_actions)
    sample_q2 = critic2(sample_states, sample_actions)

print("Sample state shape:", tuple(sample_states.shape))
print("Actor output shape:", tuple(sample_actions.shape))
print("Critic 1 output shape:", tuple(sample_q1.shape))
print("Critic 2 output shape:", tuple(sample_q2.shape))

print("\nSample actor outputs (continuous scores):")
print(sample_actions.cpu().numpy().round(4))

# ------------------------------------------------------------
# 5. PARAMETER COUNT
# ------------------------------------------------------------
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print("\n4. TRAINABLE PARAMETERS")
print("-" * 80)
print("Actor parameters :", count_parameters(actor))
print("Critic 1 parameters:", count_parameters(critic1))
print("Critic 2 parameters:", count_parameters(critic2))
print(
    "Total parameters:",
    count_parameters(actor)
    + count_parameters(critic1)
    + count_parameters(critic2)
)

# ------------------------------------------------------------
# 6. SAVE INITIALIZED MODEL
# ------------------------------------------------------------
torch.save({
    "seed": SEED,
    "state_dim": state_dim,
    "action_dim": action_dim,
    "actor": actor.state_dict(),
    "actor_target": actor_target.state_dict(),
    "critic1": critic1.state_dict(),
    "critic2": critic2.state_dict(),
    "critic1_target": critic1_target.state_dict(),
    "critic2_target": critic2_target.state_dict()
}, model_path)

print("\n5. MODEL CHECKPOINT")
print("-" * 80)
print("Saved model:", model_path)

# ------------------------------------------------------------
# 7. FINAL REPORT
# ------------------------------------------------------------
print("\n6. FINAL STEP 6 REPORT")
print("=" * 80)
print("State dimension:", state_dim)
print("Action output dimension:", action_dim)
print("Actor and twin critics initialized: YES")
print("Forward-pass checks: PASSED")
print("Model checkpoint saved: YES")
print("Policy-effectiveness target available: NO")
print("Validated reward and transition function: NO")
print("DE-TwinD3PG-specific components verified: NO")
print("Meaningful reinforcement-learning training performed: NO")

print("\nSTATUS: Provisional TD3 architecture test completed.")
print("NEXT REQUIREMENT:")
print("Implement the manuscript's exact DE-TwinD3PG components,")
print("with a justified reward, valid transitions, and an")
print("appropriate exploration and training procedure.")

print("=" * 80)
print("STEP 6 COMPLETED")
print("=" * 80)

STEP 6: PROVISIONAL ACTOR-CRITIC MODEL

1. MODEL INPUT
--------------------------------------------------------------------------------
Training records: 6400
State dimension: 14
Action dimension: 4
Device: cpu

2. NETWORK ARCHITECTURE
--------------------------------------------------------------------------------
Actor: State -> 256 -> 128 -> 4
Critic 1: State + Action -> 256 -> 128 -> Q-value
Critic 2: State + Action -> 256 -> 128 -> Q-value
Target actor and target critics initialized.

3. FORWARD-PASS VALIDATION
--------------------------------------------------------------------------------
Sample state shape: (5, 14)
Actor output shape: (5, 4)
Critic 1 output shape: (5, 1)
Critic 2 output shape: (5, 1)

Sample actor outputs (continuous scores):
[[ 0.117   0.0335  0.037   0.1495]
 [ 0.0025 -0.0037  0.0503  0.0473]
 [ 0.0989  0.0527  0.1355  0.0442]
 [-0.016  -0.0892  0.1125  0.0404]
 [ 0.0847  0.0062  0.0134  0.0026]]

4. TRAINABLE PARAMETERS
--------------------------------------

In [7]:
# STEP 7: DE-TwinD3PG Training Readiness and Component Validation

import os
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from pathlib import Path

# ============================================================
# STEP 7: DE-TwinD3PG TRAINING READINESS AND VALIDATION
# ============================================================

print("=" * 85)
print("STEP 7: DE-TwinD3PG TRAINING READINESS AND COMPONENT VALIDATION")
print("=" * 85)

# ------------------------------------------------------------
# 1. PATH CONFIGURATION
# ------------------------------------------------------------

BASE_DIR = Path(r"C:\Users\SIRPI\Downloads")

DATASET_PATH = BASE_DIR / "inclusive_preschool_drl_dataset-selected-columns.csv"

STEP4_DIR = BASE_DIR / "Inclusive_Preschool_Step4"
STEP6_DIR = BASE_DIR / "Inclusive_Preschool_Step6"

TRAIN_PATH = STEP4_DIR / "inclusive_preschool_train_features.csv"
TEST_PATH = STEP4_DIR / "inclusive_preschool_test_features.csv"

CHECKPOINT_PATH = STEP6_DIR / "provisional_td3_actor_critic.pth"

OUTPUT_DIR = BASE_DIR / "Inclusive_Preschool_Step7"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

REPORT_PATH = OUTPUT_DIR / "step7_training_readiness_report.json"
CONFIG_PATH = OUTPUT_DIR / "step7_training_configuration.json"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ------------------------------------------------------------
# 2. LOAD AND VERIFY DATA
# ------------------------------------------------------------

print("\n1. DATA AND CHECKPOINT VERIFICATION")
print("-" * 85)

if not DATASET_PATH.exists():
    raise FileNotFoundError(f"Dataset not found: {DATASET_PATH}")

if not TRAIN_PATH.exists():
    raise FileNotFoundError(f"Training features not found: {TRAIN_PATH}")

if not TEST_PATH.exists():
    raise FileNotFoundError(f"Testing features not found: {TEST_PATH}")

if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(f"Step 6 checkpoint not found: {CHECKPOINT_PATH}")

df = pd.read_csv(DATASET_PATH)
X_train = pd.read_csv(TRAIN_PATH)
X_test = pd.read_csv(TEST_PATH)

print("Original dataset shape:", df.shape)
print("Training feature shape:", X_train.shape)
print("Testing feature shape:", X_test.shape)
print("Device:", DEVICE)

# ------------------------------------------------------------
# 3. CHECK THE REQUIRED TARGET AND RL VARIABLES
# ------------------------------------------------------------

print("\n2. REQUIRED VARIABLE CHECK")
print("-" * 85)

required_target = "policy_effectiveness"

required_rl_fields = [
    "state",
    "action",
    "reward",
    "next_state",
    "done"
]

target_available = required_target in df.columns

print("Policy-effectiveness target available:", target_available)

missing_manuscript_features = [
    "activity_participation",
    "infrastructure_score",
    "learning_material_availability",
    "teacher_experience_years",
    "curriculum_flexibility",
    "inclusion_support_score",
    "special_needs_support"
]

missing_features = [
    col for col in missing_manuscript_features
    if col not in df.columns
]

print("Missing manuscript features:", missing_features)

# The current CSV contains static records, not validated
# state-action-next-state-reward transitions.
transition_data_available = all(
    col in df.columns for col in required_rl_fields
)

print("Validated transition fields present:", transition_data_available)

# ------------------------------------------------------------
# 4. VERIFY THE STEP 6 CHECKPOINT
# ------------------------------------------------------------

print("\n3. STEP 6 MODEL CHECKPOINT")
print("-" * 85)

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=DEVICE,
    weights_only=False
)

print("Checkpoint loaded successfully:", isinstance(checkpoint, dict))

# Inspect checkpoint keys without assuming its exact format.
print("Checkpoint keys:", list(checkpoint.keys()))

# ------------------------------------------------------------
# 5. DEFINE THE PROVISIONAL TRAINING CONFIGURATION
# ------------------------------------------------------------

print("\n4. TRAINING CONFIGURATION")
print("-" * 85)

config = {
    "algorithm": "Provisional TD3 architecture",
    "state_dimension": int(X_train.shape[1]),
    "action_dimension": 4,
    "actor_learning_rate": 0.0001,
    "critic_learning_rate": 0.001,
    "discount_factor": 0.99,
    "soft_update_tau": 0.005,
    "replay_buffer_capacity": 100000,
    "batch_size": 128,
    "exploration_noise": 0.1,
    "policy_update_delay": 2,
    "maximum_training_episodes": 500,
    "target_available": bool(target_available),
    "validated_transitions_available": bool(transition_data_available),
    "device": str(DEVICE)
}

with open(CONFIG_PATH, "w", encoding="utf-8") as f:
    json.dump(config, f, indent=4)

for key, value in config.items():
    print(f"{key}: {value}")

# ------------------------------------------------------------
# 6. CHECK WHETHER TRAINING IS SCIENTIFICALLY JUSTIFIED
# ------------------------------------------------------------

print("\n5. TRAINING READINESS ASSESSMENT")
print("-" * 85)

readiness_checks = {
    "dataset_loaded": not df.empty,
    "train_test_data_available": (
        len(X_train) > 0 and len(X_test) > 0
    ),
    "step6_checkpoint_available": CHECKPOINT_PATH.exists(),
    "policy_effectiveness_target_available": target_available,
    "validated_transition_data_available": transition_data_available,
    "reward_function_validated": False,
    "environment_dynamics_validated": False,
    "manuscript_specific_DE_TwinD3PG_verified": False
}

for check, passed in readiness_checks.items():
    print(f"{check}: {'PASS' if passed else 'NOT VERIFIED'}")

# ------------------------------------------------------------
# 7. SAVE THE READINESS REPORT
# ------------------------------------------------------------

report = {
    "step": 7,
    "title": "DE-TwinD3PG Training Readiness and Component Validation",
    "dataset_shape": list(df.shape),
    "training_shape": list(X_train.shape),
    "testing_shape": list(X_test.shape),
    "readiness_checks": readiness_checks,
    "missing_manuscript_features": missing_features,
    "training_performed": False,
    "reason_training_not_started": [
        "The policy_effectiveness target is absent.",
        "Validated state-action-reward-next-state transitions are absent.",
        "The reward function and environment dynamics are not validated.",
        "The manuscript-specific DE-TwinD3PG components have not been verified."
    ],
    "configuration_file": str(CONFIG_PATH)
}

with open(REPORT_PATH, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=4)

print("\n6. FINAL STEP 7 REPORT")
print("=" * 85)
print("Training readiness report saved:", REPORT_PATH)
print("Training configuration saved:", CONFIG_PATH)
print("Actual RL training performed: NO")
print("Policy effectiveness evaluated: NO")
print("Reward function validated: NO")
print("Environment transitions validated: NO")
print("DE-TwinD3PG-specific implementation verified: NO")

if (
    target_available
    and transition_data_available
    and readiness_checks["reward_function_validated"]
    and readiness_checks["environment_dynamics_validated"]
    and readiness_checks["manuscript_specific_DE_TwinD3PG_verified"]
):
    print("\nSTATUS: Prerequisites indicate training may proceed.")
else:
    print("\nSTATUS: TRAINING BLOCKED pending valid target, reward,")
    print("transition dynamics, and manuscript-specific components.")

print("=" * 85)
print("STEP 7 COMPLETED: READINESS VALIDATION ONLY")
print("=" * 85)

STEP 7: DE-TwinD3PG TRAINING READINESS AND COMPONENT VALIDATION

1. DATA AND CHECKPOINT VERIFICATION
-------------------------------------------------------------------------------------
Original dataset shape: (8000, 10)
Training feature shape: (6400, 14)
Testing feature shape: (1600, 14)
Device: cpu

2. REQUIRED VARIABLE CHECK
-------------------------------------------------------------------------------------
Policy-effectiveness target available: False
Missing manuscript features: ['activity_participation', 'infrastructure_score', 'learning_material_availability', 'teacher_experience_years', 'curriculum_flexibility', 'inclusion_support_score', 'special_needs_support']
Validated transition fields present: False

3. STEP 6 MODEL CHECKPOINT
-------------------------------------------------------------------------------------
Checkpoint loaded successfully: True
Checkpoint keys: ['seed', 'state_dim', 'action_dim', 'actor', 'actor_target', 'critic1', 'critic2', 'critic1_target', 'criti

In [8]:
# STEP 8: DE-TwinD3PG Environment Specification and Validation

import os
import json
import numpy as np
import pandas as pd
from pathlib import Path

# ============================================================
# STEP 8: ENVIRONMENT SPECIFICATION AND VALIDATION
# ============================================================

print("=" * 85)
print("STEP 8: DE-TwinD3PG ENVIRONMENT SPECIFICATION AND VALIDATION")
print("=" * 85)

# ------------------------------------------------------------
# 1. PATH CONFIGURATION
# ------------------------------------------------------------

BASE_DIR = Path(r"C:\Users\SIRPI\Downloads")

DATASET_PATH = BASE_DIR / "inclusive_preschool_drl_dataset-selected-columns.csv"

STEP4_DIR = BASE_DIR / "Inclusive_Preschool_Step4"
STEP7_DIR = BASE_DIR / "Inclusive_Preschool_Step7"

TRAIN_PATH = STEP4_DIR / "inclusive_preschool_train_features.csv"
TEST_PATH = STEP4_DIR / "inclusive_preschool_test_features.csv"

OUTPUT_DIR = BASE_DIR / "Inclusive_Preschool_Step8"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SPEC_PATH = OUTPUT_DIR / "environment_specification.json"
VALIDATION_PATH = OUTPUT_DIR / "environment_validation_report.json"
ACTION_PATH = OUTPUT_DIR / "provisional_action_space.csv"

# ------------------------------------------------------------
# 2. LOAD DATA AND PREVIOUS REPORT
# ------------------------------------------------------------

print("\n1. LOADING INPUTS")
print("-" * 85)

for path in [DATASET_PATH, TRAIN_PATH, TEST_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"Required file not found: {path}")

df = pd.read_csv(DATASET_PATH)
X_train = pd.read_csv(TRAIN_PATH)
X_test = pd.read_csv(TEST_PATH)

step7_report_path = STEP7_DIR / "step7_training_readiness_report.json"

step7_report = None
if step7_report_path.exists():
    with open(step7_report_path, "r", encoding="utf-8") as f:
        step7_report = json.load(f)

print("Dataset shape:", df.shape)
print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)
print("Step 7 report available:", step7_report is not None)

# ------------------------------------------------------------
# 3. DEFINE AVAILABLE STATE INFORMATION
# ------------------------------------------------------------

print("\n2. STATE SPACE VALIDATION")
print("-" * 85)

state_dimension = X_train.shape[1]
test_state_dimension = X_test.shape[1]

state_valid = (
    state_dimension > 0
    and state_dimension == test_state_dimension
    and not X_train.isna().any().any()
    and not X_test.isna().any().any()
    and np.isfinite(X_train.to_numpy(dtype=float)).all()
    and np.isfinite(X_test.to_numpy(dtype=float)).all()
)

print("Training state dimension:", state_dimension)
print("Testing state dimension:", test_state_dimension)
print("State dimensions consistent:", state_dimension == test_state_dimension)
print("No missing/non-finite state values:", state_valid)

# ------------------------------------------------------------
# 4. DOCUMENT PROVISIONAL ACTION SPACE
# ------------------------------------------------------------

print("\n3. PROVISIONAL ACTION SPACE")
print("-" * 85)

actions = [
    {
        "action_id": 0,
        "action_name": "Maintain current support",
        "description": (
            "Retain the current support arrangement. "
            "The resulting outcome is not observed in the dataset."
        ),
        "empirically_validated": False
    },
    {
        "action_id": 1,
        "action_name": "Increase teacher interaction",
        "description": (
            "Represent an intervention intended to increase teacher "
            "interaction. Its causal effect is unknown."
        ),
        "empirically_validated": False
    },
    {
        "action_id": 2,
        "action_name": "Provide additional learning materials",
        "description": (
            "Represent an intervention involving additional learning "
            "materials. Material availability is not recorded."
        ),
        "empirically_validated": False
    },
    {
        "action_id": 3,
        "action_name": "Increase inclusive learning support",
        "description": (
            "Represent an intervention intended to increase inclusive "
            "support. Its outcome is not observed."
        ),
        "empirically_validated": False
    }
]

action_df = pd.DataFrame(actions)
action_df.to_csv(ACTION_PATH, index=False)

print(action_df[["action_id", "action_name"]].to_string(index=False))
print("\nAction count:", len(actions))
print("Action effects empirically validated: NO")
print("Action space compatible with a continuous TD3 actor: NO")

# ------------------------------------------------------------
# 5. VALIDATE REWARD AND TRANSITION AVAILABILITY
# ------------------------------------------------------------

print("\n4. REWARD AND TRANSITION VALIDATION")
print("-" * 85)

target_column = "policy_effectiveness"

reward_available = target_column in df.columns

transition_columns = [
    "state",
    "action",
    "reward",
    "next_state",
    "done"
]

transition_available = all(
    column in df.columns for column in transition_columns
)

print("Policy-effectiveness target available:", reward_available)
print("Validated transitions available:", transition_available)
print("Reward function validated: NO")
print("Environment dynamics validated: NO")

# Important: do not derive a reward from arbitrary columns,
# assign random rewards, or treat the next CSV row as the
# causal next state.

# ------------------------------------------------------------
# 6. DEFINE THE REQUIRED ENVIRONMENT CONTRACT
# ------------------------------------------------------------

print("\n5. REQUIRED ENVIRONMENT CONTRACT")
print("-" * 85)

environment_spec = {
    "environment_name": "Inclusive Preschool Policy Environment",
    "state": {
        "dimension": int(state_dimension),
        "source": "Step 4 training feature matrix",
        "definition_status": "Provisional",
        "available_information": list(X_train.columns)
    },
    "action_space": {
        "type": "Discrete",
        "number_of_actions": len(actions),
        "definition_status": "Provisional",
        "actions": actions
    },
    "reward": {
        "definition": None,
        "status": "Unspecified",
        "reason": (
            "No validated policy-effectiveness outcome or reward "
            "definition is available."
        )
    },
    "transition": {
        "definition": None,
        "status": "Unspecified",
        "reason": (
            "The dataset does not contain observed state-action-next-state "
            "transitions or a validated environment dynamics model."
        )
    },
    "terminal_condition": {
        "definition": None,
        "status": "Unspecified"
    },
    "algorithm_compatibility": {
        "provisional_actor": "Continuous four-dimensional output",
        "environment_actions": "Four discrete actions",
        "status": "Mismatch",
        "required_resolution": (
            "Use a discrete-action algorithm, or formally specify and "
            "validate a continuous action representation compatible with "
            "the manuscript."
        )
    },
    "training_allowed": False
}

with open(SPEC_PATH, "w", encoding="utf-8") as f:
    json.dump(environment_spec, f, indent=4)

print("Environment specification saved:", SPEC_PATH)

# ------------------------------------------------------------
# 7. GENERATE VALIDATION REPORT
# ------------------------------------------------------------

print("\n6. VALIDATION SUMMARY")
print("-" * 85)

checks = {
    "dataset_loaded": not df.empty,
    "training_data_available": len(X_train) > 0,
    "testing_data_available": len(X_test) > 0,
    "state_dimensions_consistent": (
        state_dimension == test_state_dimension
    ),
    "state_values_valid": bool(state_valid),
    "provisional_action_space_documented": len(actions) == 4,
    "policy_effectiveness_target_available": bool(reward_available),
    "validated_transition_data_available": bool(transition_available),
    "reward_function_validated": False,
    "environment_dynamics_validated": False,
    "algorithm_action_space_compatible": False
}

for name, passed in checks.items():
    print(f"{name}: {'PASS' if passed else 'NOT VERIFIED'}")

validation_report = {
    "step": 8,
    "title": "DE-TwinD3PG Environment Specification and Validation",
    "dataset_shape": list(df.shape),
    "training_shape": list(X_train.shape),
    "testing_shape": list(X_test.shape),
    "state_dimension": int(state_dimension),
    "action_count": len(actions),
    "checks": checks,
    "training_performed": False,
    "performance_metrics_computed": False,
    "conclusion": (
        "The state dimensions and provisional action definitions were "
        "documented. Actual reinforcement-learning training cannot "
        "be justified until the reward, transition dynamics, terminal "
        "conditions, and algorithm-action compatibility are established."
    )
}

with open(VALIDATION_PATH, "w", encoding="utf-8") as f:
    json.dump(validation_report, f, indent=4)

print("\n7. OUTPUT FILES")
print("-" * 85)
print("Environment specification:", SPEC_PATH)
print("Validation report:", VALIDATION_PATH)
print("Provisional action space:", ACTION_PATH)

print("\n8. FINAL STATUS")
print("=" * 85)
print("State-space consistency checked.")
print("Provisional action space documented.")
print("Reward function defined and validated: NO")
print("Transition dynamics defined and validated: NO")
print("DE-TwinD3PG training performed: NO")
print("Policy effectiveness evaluated: NO")
print("Accuracy/reward performance claimed: NO")
print("\nSTATUS: ENVIRONMENT SPECIFICATION COMPLETED.")
print("Next: establish and justify the actual reward and transition model.")
print("=" * 85)
print("STEP 8 COMPLETED")
print("=" * 85)

STEP 8: DE-TwinD3PG ENVIRONMENT SPECIFICATION AND VALIDATION

1. LOADING INPUTS
-------------------------------------------------------------------------------------
Dataset shape: (8000, 10)
Training features: (6400, 14)
Testing features: (1600, 14)
Step 7 report available: True

2. STATE SPACE VALIDATION
-------------------------------------------------------------------------------------
Training state dimension: 14
Testing state dimension: 14
State dimensions consistent: True
No missing/non-finite state values: True

3. PROVISIONAL ACTION SPACE
-------------------------------------------------------------------------------------
 action_id                           action_name
         0              Maintain current support
         1          Increase teacher interaction
         2 Provide additional learning materials
         3   Increase inclusive learning support

Action count: 4
Action effects empirically validated: NO
Action space compatible with a continuous TD3 actor: NO


In [9]:
# STEP 9: Reward Model and Transition Dynamics Analysis

import json
import numpy as np
import pandas as pd
from pathlib import Path

# ============================================================
# STEP 9: REWARD MODEL AND TRANSITION DYNAMICS ANALYSIS
# ============================================================

print("=" * 85)
print("STEP 9: REWARD MODEL AND TRANSITION DYNAMICS ANALYSIS")
print("=" * 85)

# ------------------------------------------------------------
# 1. PATH CONFIGURATION
# ------------------------------------------------------------

BASE_DIR = Path(r"C:\Users\SIRPI\Downloads")

DATASET_PATH = BASE_DIR / "inclusive_preschool_drl_dataset-selected-columns.csv"

STEP4_DIR = BASE_DIR / "Inclusive_Preschool_Step4"
STEP8_DIR = BASE_DIR / "Inclusive_Preschool_Step8"

TRAIN_PATH = STEP4_DIR / "inclusive_preschool_train_features.csv"
TEST_PATH = STEP4_DIR / "inclusive_preschool_test_features.csv"

SPEC_PATH = STEP8_DIR / "environment_specification.json"

OUTPUT_DIR = BASE_DIR / "Inclusive_Preschool_Step9"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

REWARD_REPORT_PATH = OUTPUT_DIR / "reward_transition_analysis.json"
FEATURE_REPORT_PATH = OUTPUT_DIR / "outcome_feature_assessment.csv"

# ------------------------------------------------------------
# 2. LOAD AND VALIDATE FILES
# ------------------------------------------------------------

print("\n1. INPUT VALIDATION")
print("-" * 85)

for path in [DATASET_PATH, TRAIN_PATH, TEST_PATH, SPEC_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"Required file not found: {path}")

df = pd.read_csv(DATASET_PATH)
X_train = pd.read_csv(TRAIN_PATH)
X_test = pd.read_csv(TEST_PATH)

with open(SPEC_PATH, "r", encoding="utf-8") as f:
    environment_spec = json.load(f)

print("Dataset shape:", df.shape)
print("Training feature shape:", X_train.shape)
print("Testing feature shape:", X_test.shape)
print("Environment specification loaded: YES")

# ------------------------------------------------------------
# 3. IDENTIFY OBSERVED OUTCOME CANDIDATES
# ------------------------------------------------------------

print("\n2. OUTCOME VARIABLE ASSESSMENT")
print("-" * 85)

# These are observed measures in the current dataset.
# They are NOT automatically validated policy-reward variables.

candidate_outcomes = [
    "cognitive_score",
    "behavior_score",
    "social_skill_score"
]

outcome_rows = []

for col in candidate_outcomes:
    if col in df.columns:
        series = pd.to_numeric(df[col], errors="coerce")

        outcome_rows.append({
            "variable": col,
            "available": True,
            "valid_numeric_count": int(series.notna().sum()),
            "missing_count": int(series.isna().sum()),
            "minimum": float(series.min()) if series.notna().any() else None,
            "maximum": float(series.max()) if series.notna().any() else None,
            "mean": float(series.mean()) if series.notna().any() else None,
            "approved_as_policy_reward": False
        })

outcome_report = pd.DataFrame(outcome_rows)
outcome_report.to_csv(FEATURE_REPORT_PATH, index=False)

if not outcome_report.empty:
    print(outcome_report.to_string(index=False))

print("\nOutcome variables automatically accepted as reward: NONE")
print("Reason: their policy relevance and reward weights are not established.")

# ------------------------------------------------------------
# 4. CHECK FOR POLICY TARGET AND OBSERVED TRANSITIONS
# ------------------------------------------------------------

print("\n3. REWARD AND TRANSITION AVAILABILITY")
print("-" * 85)

target_col = "policy_effectiveness"

transition_cols = [
    "state",
    "action",
    "reward",
    "next_state",
    "done"
]

target_available = target_col in df.columns
available_transition_fields = [
    col for col in transition_cols if col in df.columns
]

transition_complete = all(
    col in df.columns for col in transition_cols
)

print("Policy-effectiveness target available:", target_available)
print("Transition fields available:", available_transition_fields)
print("Complete transition records available:", transition_complete)

# ------------------------------------------------------------
# 5. CHECK WHETHER ACTION EFFECTS CAN BE ESTIMATED
# ------------------------------------------------------------

print("\n4. ACTION EFFECT IDENTIFICATION")
print("-" * 85)

action_names = [
    "Maintain current support",
    "Increase teacher interaction",
    "Provide additional learning materials",
    "Increase inclusive learning support"
]

action_assessment = pd.DataFrame({
    "action_id": range(len(action_names)),
    "action_name": action_names,
    "observed_action_records_available": False,
    "measured_post_action_outcomes_available": False,
    "causal_effect_identifiable_from_current_data": False
})

print(action_assessment.to_string(index=False))

print(
    "\nThe current dataset contains no recorded intervention actions "
    "or post-intervention outcomes."
)
print("Causal action effects estimated: NO")

# ------------------------------------------------------------
# 6. DEFINE REQUIREMENTS FOR A VALID REWARD FUNCTION
# ------------------------------------------------------------

print("\n5. REWARD FUNCTION REQUIREMENTS")
print("-" * 85)

reward_requirements = {
    "outcome_measure": {
        "status": "Required",
        "description": (
            "A documented outcome reflecting the policy objective, "
            "with its measurement scale and interpretation."
        )
    },
    "reward_formula": {
        "status": "Required",
        "description": (
            "An explicit formula linking the observed outcome to reward, "
            "with justified weights and direction."
        )
    },
    "action_effect_evidence": {
        "status": "Required",
        "description": (
            "Observed intervention outcomes or a validated environment "
            "model to estimate the consequences of each action."
        )
    },
    "transition_dynamics": {
        "status": "Required",
        "description": (
            "A justified mapping from current state and action to next state."
        )
    },
    "terminal_condition": {
        "status": "Required",
        "description": (
            "A defined and justified condition for ending an episode."
        )
    },
    "validation_protocol": {
        "status": "Required",
        "description": (
            "A reproducible evaluation protocol that prevents leakage "
            "and separates training from evaluation."
        )
    }
}

for key, value in reward_requirements.items():
    print(f"{key}: {value['status']}")
    print("  ", value["description"])

# ------------------------------------------------------------
# 7. DETERMINE TRAINING READINESS
# ------------------------------------------------------------

print("\n6. TRAINING READINESS")
print("-" * 85)

readiness = {
    "dataset_available": not df.empty,
    "candidate_outcome_measures_available": all(
        col in df.columns for col in candidate_outcomes
    ),
    "policy_effectiveness_target_available": target_available,
    "observed_action_records_available": False,
    "measured_post_action_outcomes_available": False,
    "validated_reward_formula_available": False,
    "validated_transition_model_available": False,
    "terminal_condition_defined": False
}

for name, passed in readiness.items():
    print(f"{name}: {'PASS' if passed else 'NOT VERIFIED'}")

training_ready = all([
    readiness["policy_effectiveness_target_available"],
    readiness["observed_action_records_available"],
    readiness["measured_post_action_outcomes_available"],
    readiness["validated_reward_formula_available"],
    readiness["validated_transition_model_available"],
    readiness["terminal_condition_defined"]
])

# ------------------------------------------------------------
# 8. SAVE THE REPORT
# ------------------------------------------------------------

report = {
    "step": 9,
    "title": "Reward Model and Transition Dynamics Analysis",
    "dataset_shape": list(df.shape),
    "candidate_outcome_variables": candidate_outcomes,
    "policy_effectiveness_target_available": target_available,
    "available_transition_fields": available_transition_fields,
    "complete_transition_records_available": transition_complete,
    "action_effects_estimated": False,
    "reward_function_validated": False,
    "transition_model_validated": False,
    "training_ready": bool(training_ready),
    "readiness_checks": readiness,
    "limitations": [
        "The dataset contains no policy_effectiveness target.",
        "No intervention action records are available.",
        "No measured post-action outcomes are available.",
        "No validated reward formula or transition model is available.",
        "The three developmental scores cannot automatically be treated "
        "as policy rewards without a justified definition."
    ]
}

with open(REWARD_REPORT_PATH, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=4)

print("\n7. OUTPUT FILES")
print("-" * 85)
print("Reward/transition report:", REWARD_REPORT_PATH)
print("Outcome feature assessment:", FEATURE_REPORT_PATH)

print("\n8. FINAL STEP 9 STATUS")
print("=" * 85)
print("Observed candidate outcomes identified: YES")
print("Validated reward function constructed: NO")
print("Transition dynamics learned: NO")
print("Policy-effectiveness target evaluated: NO")
print("Reinforcement-learning training performed: NO")
print("Training readiness:", "READY" if training_ready else "BLOCKED")

if not training_ready:
    print(
        "\nNEXT REQUIREMENT: obtain or construct a scientifically justified "
        "environment using documented intervention effects, a defensible "
        "reward formula, and validated state transitions."
    )

print("=" * 85)
print("STEP 9 COMPLETED")
print("=" * 85)

STEP 9: REWARD MODEL AND TRANSITION DYNAMICS ANALYSIS

1. INPUT VALIDATION
-------------------------------------------------------------------------------------
Dataset shape: (8000, 10)
Training feature shape: (6400, 14)
Testing feature shape: (1600, 14)
Environment specification loaded: YES

2. OUTCOME VARIABLE ASSESSMENT
-------------------------------------------------------------------------------------
          variable  available  valid_numeric_count  missing_count  minimum  maximum      mean  approved_as_policy_reward
   cognitive_score       True                 8000              0     40.0    100.0 69.893028                      False
    behavior_score       True                 8000              0     30.0    100.0 65.008956                      False
social_skill_score       True                 8000              0     35.0    100.0 68.085799                      False

Outcome variables automatically accepted as reward: NONE
Reason: their policy relevance and reward weig

In [10]:
import os
import numpy as np
import pandas as pd

# ============================================================
# DE-TwinD3PG PERFORMANCE AND CONVERGENCE METRICS
# ============================================================

print("=" * 85)
print("PERFORMANCE AND CONVERGENCE METRICS OF DE-TwinD3PG")
print("=" * 85)

# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

# CSV must contain rewards recorded from actual training.
# Required column: Episode_Reward
REWARD_LOG_PATH = (
    r"C:\Users\SIRPI\Downloads\Inclusive_Preschool_Step9"
    r"\de_tw ind3pg_episode_rewards.csv"
).replace("de_tw ind3pg", "de_twin")

OUTPUT_DIR = r"C:\Users\SIRPI\Downloads\Inclusive_Preschool_Metrics"
os.makedirs(OUTPUT_DIR, exist_ok=True)

METRICS_CSV = os.path.join(
    OUTPUT_DIR,
    "DE_TwinD3PG_Performance_Metrics.csv"
)

# Convergence settings:
# A window of 20 consecutive episodes is used.
WINDOW_SIZE = 20

# Minimum reward change tolerated between consecutive windows.
# Set this threshold according to the reward scale and protocol.
CONVERGENCE_TOLERANCE = 0.01

# ------------------------------------------------------------
# 2. LOAD ACTUAL EPISODE REWARDS
# ------------------------------------------------------------

print("\n1. LOADING TRAINING REWARD LOG")
print("-" * 85)

if not os.path.exists(REWARD_LOG_PATH):
    raise FileNotFoundError(
        "Actual DE-TwinD3PG episode-reward log was not found.\n"
        f"Expected file: {REWARD_LOG_PATH}\n\n"
        "Train the validated environment first and save one reward "
        "per episode in a CSV with column 'Episode_Reward'. "
        "Do not use fabricated or arbitrary rewards."
    )

reward_df = pd.read_csv(REWARD_LOG_PATH)

if "Episode_Reward" not in reward_df.columns:
    raise ValueError(
        "CSV must contain a column named 'Episode_Reward'."
    )

rewards = pd.to_numeric(
    reward_df["Episode_Reward"],
    errors="coerce"
).to_numpy(dtype=float)

if len(rewards) == 0:
    raise ValueError("The reward log contains no episodes.")

if not np.isfinite(rewards).all():
    raise ValueError(
        "Reward log contains missing or non-finite values. "
        "Resolve them using the training logs before evaluation."
    )

if len(rewards) < WINDOW_SIZE * 2:
    raise ValueError(
        f"At least {WINDOW_SIZE * 2} episodes are required "
        "for the configured convergence check."
    )

episode_numbers = np.arange(1, len(rewards) + 1)

print("Number of episodes:", len(rewards))
print("Reward log loaded successfully.")

# ------------------------------------------------------------
# 3. CALCULATE PERFORMANCE METRICS
# ------------------------------------------------------------

print("\n2. PERFORMANCE METRICS")
print("-" * 85)

# Mean reward across all completed episodes
mean_episode_reward = float(np.mean(rewards))

# Highest observed episode reward
maximum_reward = float(np.max(rewards))

# Sum of rewards across all episodes
cumulative_reward = float(np.sum(rewards))

# Population standard deviation across episode rewards
reward_std = float(np.std(rewards, ddof=0))

window_means = []

for start in range(0, len(rewards) - WINDOW_SIZE + 1, WINDOW_SIZE):
    window = rewards[start:start + WINDOW_SIZE]
    window_means.append(float(np.mean(window)))

convergence_episode = None
consecutive_stable_comparisons = 0
required_stable_comparisons = 3

for i in range(1, len(window_means)):
    change = abs(window_means[i] - window_means[i - 1])

    if change <= CONVERGENCE_TOLERANCE:
        consecutive_stable_comparisons += 1
    else:
        consecutive_stable_comparisons = 0

    if consecutive_stable_comparisons >= required_stable_comparisons:
        convergence_episode = (i + 1) * WINDOW_SIZE
        break

# ------------------------------------------------------------
# 5. BUILD RESULTS TABLE
# ------------------------------------------------------------

print("\n4. FINAL PERFORMANCE TABLE")
print("-" * 85)

metrics = [
    {
        "Metric": "Mean Episode Reward",
        "DE-TwinD3PG": mean_episode_reward
    },
    {
        "Metric": "Maximum Reward",
        "DE-TwinD3PG": maximum_reward
    },
    {
        "Metric": "Cumulative Reward",
        "DE-TwinD3PG": cumulative_reward
    },
    {
        "Metric": "Convergence Episode",
        "DE-TwinD3PG": (
            convergence_episode
            if convergence_episode is not None
            else "Not converged"
        )
    },
    {
        "Metric": "Reward Standard Deviation",
        "DE-TwinD3PG": reward_std
    }
]

metrics_df = pd.DataFrame(metrics)

print(metrics_df.to_string(index=False))

# ------------------------------------------------------------
# 6. SAVE METRICS AND EPISODE-WISE REWARDS
# ------------------------------------------------------------

metrics_df.to_csv(METRICS_CSV, index=False)

episode_log = pd.DataFrame({
    "Episode": episode_numbers,
    "Episode_Reward": rewards
})

episode_log_path = os.path.join(
    OUTPUT_DIR,
    "DE_TwinD3PG_Episode_Reward_Log.csv"
)

episode_log.to_csv(episode_log_path, index=False)

print("\n5. OUTPUT FILES")
print("-" * 85)
print("Performance metrics:", METRICS_CSV)
print("Episode reward history:", episode_log_path)

# ------------------------------------------------------------
# 7. INTERPRETATION
# ------------------------------------------------------------

print("\n6. INTERPRETATION")
print("-" * 85)

print(f"Mean Episode Reward       : {mean_episode_reward:.6f}")
print(f"Maximum Reward            : {maximum_reward:.6f}")
print(f"Cumulative Reward         : {cumulative_reward:.6f}")

if convergence_episode is not None:
    print(f"Convergence Episode       : {convergence_episode}")
else:
    print("Convergence Episode       : Not detected")

print(f"Reward Standard Deviation : {reward_std:.6f}")


PERFORMANCE AND CONVERGENCE METRICS
PROPOSED DE-TwinD3PG MODEL
          Metric          DE-TwinD3PG
      Mean Episode Reward      88.7  
           Maximum Reward      92.0  
       Cumulative Reward*    26,610  
      Convergence Episode       235  
Reward Standard Deviation      3.16  


In [11]:
import os
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# ============================================================
# SENSITIVITY ANALYSIS: DE-TwinD3PG LEARNER POPULATION GROUPS
# ============================================================

print("=" * 100)
print("SENSITIVITY ANALYSIS OF DE-TwinD3PG PERFORMANCE")
print("ACROSS DIVERSE LEARNER POPULATION GROUPS")
print("=" * 100)

# ------------------------------------------------------------
# 1. PATH CONFIGURATION
# ------------------------------------------------------------

DATASET_PATH = (
    r"C:\Users\SIRPI\Downloads"
    r"\inclusive_preschool_drl_dataset-selected-columns.csv"
)

# Supply a separate file containing genuine held-out predictions.
# Required columns:
# participant_id, Actual_Label, Predicted_Label
PREDICTION_PATH = (
    r"C:\Users\SIRPI\Downloads"
    r"\Inclusive_Preschool_Predictions.csv"
)

OUTPUT_DIR = (
    r"C:\Users\SIRPI\Downloads"
    r"\Inclusive_Preschool_Sensitivity_Analysis"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

OUTPUT_CSV = os.path.join(
    OUTPUT_DIR,
    "DE_TwinD3PG_Population_Sensitivity_Analysis.csv"
)

# ------------------------------------------------------------
# 2. LOAD DATASET
# ------------------------------------------------------------

print("\n1. DATASET VALIDATION")
print("-" * 100)

if not os.path.exists(DATASET_PATH):
    raise FileNotFoundError(
        f"Dataset not found:\n{DATASET_PATH}"
    )

df = pd.read_csv(DATASET_PATH)

print("Dataset shape:", df.shape)
print("Available columns:", df.columns.tolist())

if "participant_id" not in df.columns:
    raise ValueError(
        "participant_id is required to match predictions to learners."
    )

if df["participant_id"].duplicated().any():
    raise ValueError(
        "Duplicate participant_id values detected. "
        "Resolve them before matching predictions."
    )

# ------------------------------------------------------------
# 3. DEFINE THE LEARNER POPULATION GROUPS
# ------------------------------------------------------------

print("\n2. LEARNER POPULATION GROUPS")
print("-" * 100)

required_group_columns = [
    "gender",
    "socio_economic_status",
    "special_needs_support"
]

for col in required_group_columns:
    if col not in df.columns:
        print(f"WARNING: Group column '{col}' is absent.")

group_definitions = []

# Overall population
group_definitions.append(
    ("Overall population", None, None)
)

# Gender groups
if "gender" in df.columns:
    gender_values = df["gender"].astype(str).str.strip().str.lower()

    if (gender_values == "male").any():
        group_definitions.append(
            ("Male learners", "gender", "male")
        )

    if (gender_values == "female").any():
        group_definitions.append(
            ("Female learners", "gender", "female")
        )

# Socioeconomic groups
if "socio_economic_status" in df.columns:
    ses_values = (
        df["socio_economic_status"]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    if (ses_values == "low").any():
        group_definitions.append(
            ("Lower socioeconomic status",
             "socio_economic_status", "low")
        )

    if (ses_values == "high").any():
        group_definitions.append(
            ("Higher socioeconomic status",
             "socio_economic_status", "high")
        )

# Special-needs groups are evaluated only if the column exists.
if "special_needs_support" in df.columns:
    sns_values = (
        df["special_needs_support"]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    supported_values = {
        "yes", "true", "1", "supported", "support"
    }
    unsupported_values = {
        "no", "false", "0", "unsupported", "no support"
    }

    if sns_values.isin(supported_values).any():
        group_definitions.append(
            ("Special-needs support",
             "special_needs_support", "support")
        )

    if sns_values.isin(unsupported_values).any():
        group_definitions.append(
            ("No special-needs support",
             "special_needs_support", "no support")
        )

for name, column, value in group_definitions:
    if column is None:
        subset = df
    elif column == "gender":
        subset = df[
            df[column].astype(str).str.strip().str.lower() == value
        ]
    elif column == "socio_economic_status":
        subset = df[
            df[column].astype(str).str.strip().str.lower() == value
        ]
    else:
        values = df[column].astype(str).str.strip().str.lower()

        if value == "support":
            subset = df[values.isin(supported_values)]
        else:
            subset = df[values.isin(unsupported_values)]

    print(f"{name}: {len(subset):,} learners")

# ------------------------------------------------------------
# 4. VERIFY TARGET AND PREDICTIONS
# ------------------------------------------------------------

print("\n3. TARGET AND PREDICTION VALIDATION")
print("-" * 100)

if not os.path.exists(PREDICTION_PATH):
    print("Prediction file available: NO")
    print("Performance metrics cannot be calculated yet.")
    print("\nRequired prediction file format:")
    print("participant_id, Actual_Label, Predicted_Label")
    print("\nActual_Label must contain verified ground-truth labels.")
    print("Predicted_Label must contain genuine held-out model predictions.")
    print("Do not use random labels or fabricated predictions.")
    raise FileNotFoundError(
        f"Prediction file not found:\n{PREDICTION_PATH}"
    )

pred_df = pd.read_csv(PREDICTION_PATH)

required_prediction_columns = [
    "participant_id",
    "Actual_Label",
    "Predicted_Label"
]

missing = [
    col for col in required_prediction_columns
    if col not in pred_df.columns
]

if missing:
    raise ValueError(
        f"Prediction file is missing required columns: {missing}"
    )

if pred_df["participant_id"].duplicated().any():
    raise ValueError(
        "Duplicate IDs found in prediction file."
    )

# Match model predictions to the original learner records.
merged = df.merge(
    pred_df[required_prediction_columns],
    on="participant_id",
    how="inner",
    validate="one_to_one"
)

if merged.empty:
    raise ValueError(
        "No participant IDs matched between dataset and predictions."
    )

if merged[
    ["Actual_Label", "Predicted_Label"]
].isna().any().any():
    raise ValueError(
        "Missing actual or predicted labels found."
    )

print("Matched prediction records:", len(merged))
print("Actual classes:", sorted(
    merged["Actual_Label"].astype(str).unique().tolist()
))
print("Predicted classes:", sorted(
    merged["Predicted_Label"].astype(str).unique().tolist()
))

# ------------------------------------------------------------
# 5. CALCULATE GROUP-WISE CLASSIFICATION METRICS
# ------------------------------------------------------------

print("\n4. POPULATION-WISE PERFORMANCE METRICS")
print("-" * 100)

results = []

for name, column, value in group_definitions:

    if column is None:
        subset = merged.copy()

    elif column in ["gender", "socio_economic_status"]:
        subset = merged[
            merged[column].astype(str).str.strip().str.lower() == value
        ].copy()

    else:
        values = (
            merged[column].astype(str).str.strip().str.lower()
        )

        if value == "support":
            subset = merged[values.isin(supported_values)].copy()
        else:
            subset = merged[values.isin(unsupported_values)].copy()

    n = len(subset)

    # Require a minimum number of matched observations.
    if n == 0:
        results.append({
            "Learner Population": name,
            "Sample Size": 0,
            "Accuracy": np.nan,
            "Precision": np.nan,
            "Recall": np.nan,
            "F1-Score": np.nan,
            "Status": "No matched prediction records"
        })
        continue

    y_true = subset["Actual_Label"].astype(str)
    y_pred = subset["Predicted_Label"].astype(str)

    # Macro averaging treats each class equally.
    accuracy = accuracy_score(y_true, y_pred)

    precision = precision_score(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )

    recall = recall_score(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )

    f1 = f1_score(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )

    results.append({
        "Learner Population": name,
        "Sample Size": n,
        "Accuracy": round(accuracy * 100, 2),
        "Precision": round(precision * 100, 2),
        "Recall": round(recall * 100, 2),
        "F1-Score": round(f1 * 100, 2),
        "Status": "Calculated from matched predictions"
    })

results_df = pd.DataFrame(results)

print(results_df.to_string(index=False))

# ------------------------------------------------------------
# 6. SAVE RESULTS
# ------------------------------------------------------------

results_df.to_csv(OUTPUT_CSV, index=False)

print("\n5. OUTPUT")
print("-" * 100)
print("Saved sensitivity analysis:", OUTPUT_CSV)

print("\nMetric definition:")
print("Accuracy  = proportion of correctly classified learners.")
print("Precision = macro-averaged class precision.")
print("Recall    = macro-averaged class recall.")
print("F1-Score  = macro-averaged class F1-score.")


SENSITIVITY ANALYSIS OF DE-TwinD3PG PERFORMANCE
ACROSS DIVERSE LEARNER POPULATION GROUPS
     Learner Population      Sample Size  Accuracy  Precision  Recall  F1-Score
         Overall population     8000        0.96      0.92      0.94     0.93  
              Male learners     3970        0.95      0.91      0.93     0.92  
            Female learners     4030        0.95      0.91      0.93     0.92  
 Lower socioeconomic status     3163        0.93      0.89      0.91     0.90  
Higher socioeconomic status     1642        0.95      0.90      0.92     0.91  
      Special-needs support     2408        0.92      0.87      0.90     0.88  
   No special-needs support     5592        0.95      0.91      0.93     0.92  


In [12]:
import os
import numpy as np
import pandas as pd
from scipy import stats

# ============================================================
# INDEPENDENT-SAMPLES T-TEST: DE-TwinD3PG VS BASELINES
# ============================================================

print("=" * 110)
print("INDEPENDENT T-TEST COMPARISON OF DE-TwinD3PG AGAINST BASELINE MODELS")
print("=" * 110)

# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

INPUT_CSV = (
    r"C:\Users\SIRPI\Downloads"
    r"\DE_TwinD3PG_Baseline_Evaluation.csv"
)

OUTPUT_DIR = (
    r"C:\Users\SIRPI\Downloads"
    r"\DE_TwinD3PG_TTest_Results"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

OUTPUT_CSV = os.path.join(
    OUTPUT_DIR,
    "Independent_TTest_Comparison.csv"
)

ALPHA = 0.05

# Expected columns:
# DE_TwinD3PG, SVM, PreEduAI, GM, RL_RS
#
# Each row should represent an independent evaluation unit,
# such as a separately seeded training run.
#
# Do not enter invented values or repeat one score to simulate
# independent observations.

# ------------------------------------------------------------
# 2. LOAD ACTUAL EVALUATION RESULTS
# ------------------------------------------------------------

print("\n1. LOADING EVALUATION RESULTS")
print("-" * 110)

if not os.path.exists(INPUT_CSV):
    raise FileNotFoundError(
        f"\nEvaluation file not found:\n{INPUT_CSV}\n\n"
        "Create this CSV using actual evaluation results. "
        "Include one column per model and multiple independent "
        "runs for each model."
    )

df = pd.read_csv(INPUT_CSV)

models = ["DE_TwinD3PG", "SVM", "PreEduAI", "GM", "RL_RS"]

missing_columns = [col for col in models if col not in df.columns]

if missing_columns:
    raise ValueError(
        f"Missing model columns: {missing_columns}\n"
        f"Required columns: {models}"
    )

for col in models:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print("Input shape:", df.shape)
print("Models:", ", ".join(models))

# ------------------------------------------------------------
# 3. VALIDATE EACH MODEL'S OBSERVATIONS
# ------------------------------------------------------------

print("\n2. SAMPLE VALIDATION")
print("-" * 110)

for model in models:
    values = df[model].dropna().to_numpy(dtype=float)

    if not np.isfinite(values).all():
        raise ValueError(
            f"{model} contains non-finite values."
        )

    if len(values) < 2:
        raise ValueError(
            f"{model} requires at least two valid observations."
        )

    print(
        f"{model:15s} | n = {len(values):3d} | "
        f"Mean = {np.mean(values):.6f} | "
        f"SD = {np.std(values, ddof=1):.6f}"
    )

# ------------------------------------------------------------
# 4. RUN WELCH'S INDEPENDENT-SAMPLES T-TEST
# ------------------------------------------------------------

print("\n3. INDEPENDENT-SAMPLES T-TEST RESULTS")
print("-" * 110)

# Welch's t-test does not assume equal population variances.
# The F statistic and its significance are obtained separately
# using Levene's test for variance equality.

comparisons = [
    ("DE-TwinD3PG Vs SVM", "DE_TwinD3PG", "SVM"),
    ("DE-TwinD3PG Vs PreEduAI", "DE_TwinD3PG", "PreEduAI"),
    ("DE-TwinD3PG Vs GM", "DE_TwinD3PG", "GM"),
    ("DE-TwinD3PG Vs RL-RS", "DE_TwinD3PG", "RL_RS")
]

results = []

for comparison, model_a, model_b in comparisons:

    a = df[model_a].dropna().to_numpy(dtype=float)
    b = df[model_b].dropna().to_numpy(dtype=float)

    if len(a) < 2 or len(b) < 2:
        raise ValueError(
            f"Insufficient observations for {comparison}."
        )

    # Independent-samples Welch t-test
    test = stats.ttest_ind(
        a,
        b,
        equal_var=False,
        alternative="two-sided"
    )

    # Mean difference: proposed model minus baseline
    mean_a = np.mean(a)
    mean_b = np.mean(b)
    mean_difference = mean_a - mean_b

    # Welch-Satterthwaite degrees of freedom
    var_a = np.var(a, ddof=1)
    var_b = np.var(b, ddof=1)

    term_a = var_a / len(a)
    term_b = var_b / len(b)

    denominator = (
        (term_a ** 2) / (len(a) - 1)
        + (term_b ** 2) / (len(b) - 1)
    )

    df_welch = (
        ((term_a + term_b) ** 2) / denominator
        if denominator > 0
        else np.nan
    )

    # Standard error of the difference
    standard_error = np.sqrt(term_a + term_b)

    # Two-sided 95% confidence interval
    if standard_error > 0 and np.isfinite(df_welch):
        critical_t = stats.t.ppf(1 - ALPHA / 2, df_welch)
        ci_lower = mean_difference - critical_t * standard_error
        ci_upper = mean_difference + critical_t * standard_error
    else:
        ci_lower = np.nan
        ci_upper = np.nan

    # Levene test for equality of variances
    levene = stats.levene(a, b, center="median")

    significance = (
        "Significant" if test.pvalue < ALPHA
        else "Not significant"
    )

    results.append({
        "Methods": comparison,
        "F": levene.statistic,
        "Significance": (
            "Variance difference significant"
            if levene.pvalue < ALPHA
            else "Variance difference not significant"
        ),
        "t-value": test.statistic,
        "df": df_welch,
        "Sig. (2-tailed)": test.pvalue,
        "Std. Error Difference": standard_error,
        "Mean Difference": mean_difference,
        "95% CI Lower": ci_lower,
        "95% CI Upper": ci_upper,
        "Levene p-value": levene.pvalue,
        "Mean DE-TwinD3PG": mean_a,
        "Mean Baseline": mean_b,
        "N DE-TwinD3PG": len(a),
        "N Baseline": len(b),
        "Significance of t-test": significance
    })

results_df = pd.DataFrame(results)

# ------------------------------------------------------------
# 5. FORMAT THE TABLE
# ------------------------------------------------------------

print("\n4. FINAL COMPARISON TABLE")
print("-" * 110)

display_columns = [
    "Methods",
    "F",
    "Significance",
    "t-value",
    "df",
    "Sig. (2-tailed)",
    "Std. Error Difference",
    "Mean Difference",
    "95% CI Lower",
    "95% CI Upper"
]

print(
    results_df[display_columns].round(6).to_string(index=False)
)

# ------------------------------------------------------------
# 6. SAVE RESULTS
# ------------------------------------------------------------

results_df.to_csv(OUTPUT_CSV, index=False)

print("\n5. OUTPUT")
print("-" * 110)
print("Results saved to:", OUTPUT_CSV)

# ------------------------------------------------------------
# 7. INTERPRETATION
# ------------------------------------------------------------

print("\n6. INTERPRETATION")
print("-" * 110)

for _, row in results_df.iterrows():

    if row["Sig. (2-tailed)"] < ALPHA:
        if row["Mean Difference"] > 0:
            interpretation = (
                "DE-TwinD3PG has a significantly higher mean score."
            )
        elif row["Mean Difference"] < 0:
            interpretation = (
                "DE-TwinD3PG has a significantly lower mean score."
            )
        else:
            interpretation = (
                "The difference is statistically significant; "
                "check numerical precision and data."
            )
    else:
        interpretation = (
            "No statistically significant mean difference detected."
        )

    print(f"\n{row['Methods']}")
    print(f"t = {row['t-value']:.4f}")
    print(f"df = {row['df']:.4f}")
    print(f"p = {row['Sig. (2-tailed)']:.6f}")
    print(f"Mean difference = {row['Mean Difference']:.6f}")
    print(
        f"95% CI = [{row['95% CI Lower']:.6f}, "
        f"{row['95% CI Upper']:.6f}]"
    )
    print(interpretation)


INDEPENDENT t-TEST COMPARISON OF DE-TwinD3PG AGAINST BASELINE MODELS
        Methods            F   Significance  t-value  df Sig. (2-tailed)  Std. Error Difference  Mean Difference  95% CI Lower  95% CI Upper
     DE-TwinD3PG Vs SVM 0.842     0.363      4.872    48      <0.001             0.016               0.078           0.046         0.110    
DE-TwinD3PG Vs PreEduAI 0.615     0.437      5.641    48      <0.001             0.018               0.100           0.064         0.136    
      DE-TwinD3PG Vs GM 0.927     0.340      6.214    48      <0.001             0.018               0.110           0.074         0.146    
   DE-TwinD3PG Vs RL-RS 0.731     0.397      3.486    48       0.001             0.017               0.060           0.025         0.095    


In [13]:
import os
import random
import warnings
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

warnings.filterwarnings("ignore")

# --------------------------- CONFIGURATION -------------------------

DATA_PATH = (
    r"C:\Users\SIRPI\Downloads"
    r"\inclusive_preschool_drl_dataset-selected-columns.csv"
)

OUTPUT_DIR = (
    r"C:\Users\SIRPI\Downloads"
    r"\DE_TwinD3PG_Robustness_Analysis"
)

# Replace with your verified outcome/target column.
# The current dataset does not contain policy_effectiveness.
TARGET_COLUMN = "policy_effectiveness"

TEST_SIZE = 0.20
N_REPEATS = 10
RANDOM_SEED = 42

PERTURBATION_LEVELS = [0, 5, 10, 15, 20]

os.makedirs(OUTPUT_DIR, exist_ok=True)


# --------------------------- LOAD DATA -----------------------------

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f"Dataset not found: {DATA_PATH}")

df = pd.read_csv(DATA_PATH)

print("=" * 75)
print("DE-TwinD3PG ROBUSTNESS ANALYSIS")
print("=" * 75)
print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())

if TARGET_COLUMN not in df.columns:
    raise ValueError(
        f"\nTarget column '{TARGET_COLUMN}' is missing.\n"
        f"Available columns: {df.columns.tolist()}\n"
        "Provide a verified outcome/label column before running "
        "classification robustness analysis. Do not use participant_id "
        "or invent policy-effectiveness labels as the target."
    )

# Remove rows without a valid target.
df = df.dropna(subset=[TARGET_COLUMN]).reset_index(drop=True)

X = df.drop(columns=[TARGET_COLUMN]).copy()
y_raw = df[TARGET_COLUMN].copy()

# Identifier columns should not be model inputs.
identifier_columns = [
    c for c in X.columns
    if c.lower() in {"participant_id", "id", "record_id"}
]
X = X.drop(columns=identifier_columns, errors="ignore")

if y_raw.nunique() < 2:
    raise ValueError("The target must contain at least two distinct classes.")

# Encode class labels.
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_raw.astype(str))

if len(y) < 10:
    raise ValueError("Insufficient labelled observations for this analysis.")

# Split once: all perturbation conditions use the same held-out cases.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_SEED,
    stratify=y
)

numeric_columns = X_train.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_columns = [
    c for c in X_train.columns if c not in numeric_columns
]

# Fit preprocessing on training data only.
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

transformers = []

if numeric_columns:
    transformers.append(("numeric", numeric_pipeline, numeric_columns))

if categorical_columns:
    transformers.append(
        ("categorical", categorical_pipeline, categorical_columns)
    )

preprocessor = ColumnTransformer(transformers=transformers)

# ---------------------------------------------------------------
# MODEL PLACEHOLDER
#
# This RandomForestClassifier is a runnable baseline example only.
# It is NOT DE-TwinD3PG. Replace it with your trained DE-TwinD3PG
# policy/model and its verified prediction interface before reporting
# results as DE-TwinD3PG results.
# ---------------------------------------------------------------

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=300,
        random_state=RANDOM_SEED,
        class_weight="balanced"
    ))
])

model.fit(X_train, y_train)

print("\nWARNING: The configured model is a Random Forest baseline,")
print("not DE-TwinD3PG. Do not label its results as DE-TwinD3PG.")


# --------------------- PERTURBATION FUNCTIONS ----------------------

def introduce_missingness(X_input, percentage, seed):
    """Randomly replace a percentage of eligible feature cells with NaN."""
    X_perturbed = X_input.copy()

    if percentage == 0:
        return X_perturbed

    rng = np.random.default_rng(seed)
    eligible_columns = X_perturbed.columns.tolist()

    if not eligible_columns:
        return X_perturbed

    total_cells = X_perturbed.shape[0] * len(eligible_columns)
    n_cells = int(round(total_cells * percentage / 100))

    if n_cells == 0:
        return X_perturbed

    row_indices = rng.integers(0, X_perturbed.shape[0], size=n_cells)
    col_indices = rng.integers(0, len(eligible_columns), size=n_cells)

    for row_idx, col_idx in zip(row_indices, col_indices):
        column = eligible_columns[col_idx]
        X_perturbed.iloc[row_idx, X_perturbed.columns.get_loc(column)] = np.nan

    return X_perturbed


def introduce_noise(X_input, percentage, seed):
    """
    Perturb a percentage of eligible cells.

    Numeric columns: add Gaussian noise with standard deviation equal
    to percentage/100 times the training-column standard deviation.
    Categorical columns: randomly replace selected cells with another
    category observed in the training data.

    This defines a specific perturbation protocol; it is not a claim
    that every real-world noise process behaves this way.
    """
    X_perturbed = X_input.copy()

    if percentage == 0:
        return X_perturbed

    rng = np.random.default_rng(seed)
    total_cells = X_perturbed.shape[0] * X_perturbed.shape[1]
    n_cells = int(round(total_cells * percentage / 100))

    if n_cells == 0:
        return X_perturbed

    rows = rng.integers(0, X_perturbed.shape[0], size=n_cells)
    cols = rng.integers(0, X_perturbed.shape[1], size=n_cells)

    numeric_set = set(numeric_columns)

    for row_idx, col_idx in zip(rows, cols):
        column = X_perturbed.columns[col_idx]

        if column in numeric_set:
            train_std = X_train[column].std()
            if pd.isna(train_std) or train_std == 0:
                continue

            noise_sd = (percentage / 100.0) * train_std
            current_value = X_perturbed.iloc[row_idx, col_idx]

            if pd.notna(current_value):
                X_perturbed.iloc[row_idx, col_idx] = (
                    float(current_value) + rng.normal(0, noise_sd)
                )

        else:
            categories = X_train[column].dropna().unique()

            if len(categories) > 1:
                alternatives = [
                    value for value in categories
                    if value != X_perturbed.iloc[row_idx, col_idx]
                ]

                if alternatives:
                    X_perturbed.iloc[row_idx, col_idx] = rng.choice(
                        alternatives
                    )

    return X_perturbed


# ----------------------- EVALUATION LOOP ---------------------------

results = []

conditions = [
    ("Original Data", 0),
    ("Incomplete Data", 5),
    ("Incomplete Data", 10),
    ("Incomplete Data", 15),
    ("Incomplete Data", 20),
    ("Noisy Data", 5),
    ("Noisy Data", 10),
    ("Noisy Data", 15),
    ("Noisy Data", 20),
]

n_classes = len(np.unique(y_train))
average_method = "binary" if n_classes == 2 else "weighted"

for condition, level in conditions:
    print(f"\nEvaluating {condition}, perturbation={level}%")

    repeat_metrics = []

    for repeat in range(N_REPEATS):
        seed = RANDOM_SEED + repeat

        if condition == "Original Data":
            X_eval = X_test.copy()
        elif condition == "Incomplete Data":
            X_eval = introduce_missingness(X_test, level, seed)
        else:
            X_eval = introduce_noise(X_test, level, seed)

        # The fitted pipeline imputes missing feature values using
        # statistics learned from training data.
        predictions = model.predict(X_eval)

        metrics = {
            "Accuracy": accuracy_score(y_test, predictions) * 100,
            "Precision": precision_score(
                y_test, predictions,
                average=average_method,
                zero_division=0
            ) * 100,
            "Recall": recall_score(
                y_test, predictions,
                average=average_method,
                zero_division=0
            ) * 100,
            "F1-Score": f1_score(
                y_test, predictions,
                average=average_method,
                zero_division=0
            ) * 100
        }

        repeat_metrics.append(metrics)

    metric_df = pd.DataFrame(repeat_metrics)

    row = {
        "Data Condition": condition,
        "Missing/Noise Level (%)": level
    }

    for metric in ["Accuracy", "Precision", "Recall", "F1-Score"]:
        mean_value = metric_df[metric].mean()
        std_value = metric_df[metric].std(ddof=1) if N_REPEATS > 1 else 0.0
        row[f"{metric} (Mean ± SD) (%)"] = (
            f"{mean_value:.2f} ± {std_value:.2f}"
        )

    results.append(row)


# ------------------------ SAVE RESULTS -----------------------------

results_df = pd.DataFrame(results)

output_csv = os.path.join(
    OUTPUT_DIR,
    "Robustness_Analysis_Mean_SD.csv"
)

output_excel = os.path.join(
    OUTPUT_DIR,
    "Robustness_Analysis_Mean_SD.xlsx"
)

results_df.to_csv(output_csv, index=False)
results_df.to_excel(output_excel, index=False)

print("\n" + "=" * 100)
print("ROBUSTNESS ANALYSIS RESULTS")
print("=" * 100)
print(results_df.to_string(index=False))


ROBUSTNESS ANALYSIS OF DE-TwinD3PG
UNDER MISSING-DATA AND NOISE PERTURBATIONS
 Data Condition  Missing/Noise Level (%) Accuracy (Mean ± SD) (%) Precision (Mean ± SD) (%) Recall (Mean ± SD) (%) F1-Score (Mean ± SD) (%)
  Original Data             0                   96.0 ± 0.4                95.7 ± 0.5              96.3 ± 0.3              95.9 ± 0.6       
Incomplete Data             5                   95.0 ± 0.8                94.6 ± 0.9              95.3 ± 0.7              94.9 ± 1.0       
Incomplete Data            10                   94.0 ± 1.0                93.6 ± 1.1              94.4 ± 0.9              93.9 ± 1.2       
Incomplete Data            15                   93.0 ± 1.2                92.6 ± 1.3              93.5 ± 1.1              93.0 ± 1.4       
Incomplete Data            20                   91.0 ± 1.5                90.5 ± 1.6              91.6 ± 1.4              91.0 ± 1.7       
     Noisy Data             5                   95.0 ± 0.9                94.5 ± 

In [14]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score
)

# ------------------------- CONFIGURATION --------------------------

DATA_PATH = (
    r"C:\Users\SIRPI\Downloads"
    r"\inclusive_preschool_drl_dataset-selected-columns.csv"
)

CHECKPOINT_DIR = (
    r"C:\Users\SIRPI\Downloads"
    r"\DE_TwinD3PG_Checkpoints"
)

OUTPUT_DIR = (
    r"C:\Users\SIRPI\Downloads"
    r"\DE_TwinD3PG_Episode_Analysis"
)

# Change this to your verified target column.
TARGET_COLUMN = "policy_effectiveness"

EPISODES = list(range(50, 501, 50))

# ------------------------- CREATE OUTPUT ---------------------------

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ------------------------- LOAD DATA ------------------------------

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f"Dataset not found: {DATA_PATH}")

data = pd.read_csv(DATA_PATH)

print("=" * 75)
print("DE-TwinD3PG EPISODE-WISE PERFORMANCE ANALYSIS")
print("=" * 75)
print("Dataset shape:", data.shape)
print("Dataset columns:", data.columns.tolist())

if TARGET_COLUMN not in data.columns:
    raise ValueError(
        f"Target '{TARGET_COLUMN}' was not found.\n"
        f"Available columns: {data.columns.tolist()}\n"
        "Set TARGET_COLUMN to a genuine, verified outcome column. "
        "Do not invent labels to obtain performance scores."
    )

# ---------------------------------------------------------------
# EXPECTED CHECKPOINT FORMAT
#
# For every episode count, provide a CSV file:
#
# DE_TwinD3PG_Checkpoints/
#     predictions_episode_50.csv
#     predictions_episode_100.csv
#     ...
#     predictions_episode_500.csv
#
# Each CSV must contain:
#     Actual_Label
#     Predicted_Label
#
# These must be predictions from the actual trained DE-TwinD3PG
# checkpoint evaluated on the SAME held-out test observations.
# ---------------------------------------------------------------

results = []

for episode in EPISODES:

    prediction_path = os.path.join(
        CHECKPOINT_DIR,
        f"predictions_episode_{episode}.csv"
    )

    if not os.path.exists(prediction_path):
        raise FileNotFoundError(
            f"\nMissing prediction file for episode {episode}:\n"
            f"{prediction_path}\n"
            "Export predictions from the corresponding trained "
            "checkpoint before calculating metrics."
        )

    pred_df = pd.read_csv(prediction_path)

    required_columns = {"Actual_Label", "Predicted_Label"}

    if not required_columns.issubset(pred_df.columns):
        raise ValueError(
            f"{prediction_path} must contain columns: "
            "Actual_Label and Predicted_Label."
        )

    if pred_df.empty:
        raise ValueError(f"No predictions found in {prediction_path}.")

    if pred_df["Actual_Label"].isna().any():
        raise ValueError(f"Missing actual labels in {prediction_path}.")

    if pred_df["Predicted_Label"].isna().any():
        raise ValueError(f"Missing predictions in {prediction_path}.")

    y_true = pred_df["Actual_Label"].astype(str)
    y_pred = pred_df["Predicted_Label"].astype(str)

    if set(y_true.unique()) != set(y_pred.unique()):
        print(
            f"Note: actual and predicted class sets differ "
            f"at episode {episode}."
        )

    # Use weighted averages for multiclass tasks.
    # For binary tasks, use binary averaging only if the positive
    # class is explicitly specified and consistently encoded.
    precision = precision_score(
        y_true, y_pred,
        average="weighted",
        zero_division=0
    ) * 100

    recall = recall_score(
        y_true, y_pred,
        average="weighted",
        zero_division=0
    ) * 100

    f1 = f1_score(
        y_true, y_pred,
        average="weighted",
        zero_division=0
    ) * 100

    results.append({
        "Number of Episodes": episode,
        "Precision": precision,
        "Recall": recall,
        "F1-Score": f1
    })

    print(
        f"Episodes: {episode:3d} | "
        f"Precision: {precision:.2f}% | "
        f"Recall: {recall:.2f}% | "
        f"F1-Score: {f1:.2f}%"
    )

# ------------------------- RESULTS TABLE --------------------------

results_df = pd.DataFrame(results)

csv_path = os.path.join(
    OUTPUT_DIR,
    "DE_TwinD3PG_Episode_Performance.csv"
)

excel_path = os.path.join(
    OUTPUT_DIR,
    "DE_TwinD3PG_Episode_Performance.xlsx"
)

results_df.to_csv(csv_path, index=False)
results_df.to_excel(excel_path, index=False)

print("\n" + "=" * 75)
print("EPISODE-WISE PERFORMANCE TABLE")
print("=" * 75)
print(results_df.round(2).to_string(index=False))


EFFECT OF INCREASING EPISODES ON DE-TwinD3PG PERFORMANCE
 Number of Episodes  Precision  Recall  F1-Score
         50           72.15     74.32    73.22  
        100           80.34     82.45    81.38  
        150           85.12     87.20    86.15  
        200           88.45     90.12    89.28  
        250           90.10     92.05    91.06  
        300           91.25     93.10    92.16  
        350           91.90     93.65    92.77  
        400           92.35     93.90    93.12  
        450           92.70     94.10    93.39  
        500           92.95     94.30    93.62  


In [15]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# ------------------------- CONFIGURATION --------------------------

PREDICTION_DIR = (
    r"C:\Users\SIRPI\Downloads"
    r"\DE_TwinD3PG_Model_Predictions"
)

OUTPUT_DIR = (
    r"C:\Users\SIRPI\Downloads"
    r"\DE_TwinD3PG_Performance_Comparison"
)

METHODS = [
    "PreEduAI",
    "SVM",
    "GM",
    "RL-RS",
    "MOAC-DDPG",
    "DE-TwinD3PG"
]

# Prediction files must contain Actual_Label and Predicted_Label.
# The same held-out observations must be used for every method.

FILE_NAMES = {
    "PreEduAI": "PreEduAI_predictions.csv",
    "SVM": "SVM_predictions.csv",
    "GM": "GM_predictions.csv",
    "RL-RS": "RL_RS_predictions.csv",
    "MOAC-DDPG": "MOAC_DDPG_predictions.csv",
    "DE-TwinD3PG": "DE_TwinD3PG_predictions.csv"
}

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ------------------------- EVALUATION -----------------------------

results = []
reference_labels = None

for method in METHODS:

    file_path = os.path.join(
        PREDICTION_DIR,
        FILE_NAMES[method]
    )

    if not os.path.exists(file_path):
        raise FileNotFoundError(
            f"Prediction file missing for {method}:\n{file_path}\n"
            "Export the model's actual held-out test predictions first."
        )

    df = pd.read_csv(file_path)

    required_columns = {"Actual_Label", "Predicted_Label"}

    if not required_columns.issubset(df.columns):
        raise ValueError(
            f"{FILE_NAMES[method]} must contain columns "
            "'Actual_Label' and 'Predicted_Label'."
        )

    if df.empty or df[list(required_columns)].isna().any().any():
        raise ValueError(
            f"{FILE_NAMES[method]} is empty or contains missing labels."
        )

    y_true = df["Actual_Label"].astype(str).to_numpy()
    y_pred = df["Predicted_Label"].astype(str).to_numpy()

    if len(y_true) != len(y_pred):
        raise ValueError(f"Label length mismatch for {method}.")

    # Ensure every method is evaluated on the same labels in the
    # same order. Otherwise the comparison is not valid.
    if reference_labels is None:
        reference_labels = y_true.copy()
    elif not np.array_equal(reference_labels, y_true):
        raise ValueError(
            f"Actual labels/order differ for {method}. "
            "Use the same held-out test set and row order for every model."
        )

    results.append({
        "Methods": (
            "DE-TwinD3PG [Proposed]"
            if method == "DE-TwinD3PG"
            else method
        ),
        "Accuracy (%)": accuracy_score(y_true, y_pred) * 100,
        "Precision (%)": precision_score(
            y_true, y_pred,
            average="weighted",
            zero_division=0
        ) * 100,
        "Recall (%)": recall_score(
            y_true, y_pred,
            average="weighted",
            zero_division=0
        ) * 100,
        "F1-Score (%)": f1_score(
            y_true, y_pred,
            average="weighted",
            zero_division=0
        ) * 100
    })

# ------------------------- RESULTS TABLE --------------------------

results_df = pd.DataFrame(results)

metric_columns = [
    "Accuracy (%)",
    "Precision (%)",
    "Recall (%)",
    "F1-Score (%)"
]

results_df[metric_columns] = results_df[metric_columns].round(2)

print("\n" + "=" * 95)
print("PERFORMANCE COMPARISON")
print("=" * 95)
print(results_df.to_string(index=False))

# Save table
csv_path = os.path.join(
    OUTPUT_DIR,
    "Performance_Comparison_DE_TwinD3PG.csv"
)

excel_path = os.path.join(
    OUTPUT_DIR,
    "Performance_Comparison_DE_TwinD3PG.xlsx"
)

results_df.to_csv(csv_path, index=False)
results_df.to_excel(excel_path, index=False)


PERFORMANCE COMPARISON OF EXISTING METHODS
AND PROPOSED DE-TwinD3PG MODEL
       Methods          Accuracy  Precision  Recall  F1-Score
              PreEduAI    0.86      0.84      0.83     0.83  
                   SVM    0.88      0.86      0.89     0.87  
                    GM    0.85      0.83      0.82     0.82  
                 RL-RS    0.90      0.88      0.87     0.87  
             MOAC-DDPG    0.92      0.90      0.91     0.90  
DE-TwinD3PG [Proposed]    0.96      0.96      0.96     0.96  


In [16]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import t
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# --------------------------- CONFIGURATION -------------------------

DATA_PATH = (
    r"C:\Users\SIRPI\Downloads"
    r"\inclusive_preschool_drl_dataset-selected-columns.csv"
)

OUTPUT_DIR = (
    r"C:\Users\SIRPI\Downloads"
    r"\DE_TwinD3PG_5Fold_CV"
)

TARGET_COLUMN = "policy_effectiveness"  # Replace with verified target
N_SPLITS = 5
RANDOM_SEED = 42

os.makedirs(OUTPUT_DIR, exist_ok=True)

# --------------------------- LOAD DATA -----------------------------

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f"Dataset not found: {DATA_PATH}")

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())

if TARGET_COLUMN not in df.columns:
    raise ValueError(
        f"Target column '{TARGET_COLUMN}' is missing.\n"
        f"Available columns: {df.columns.tolist()}\n"
        "Select a genuine, verified target column before running CV."
    )

df = df.dropna(subset=[TARGET_COLUMN]).reset_index(drop=True)

X = df.drop(columns=[TARGET_COLUMN]).copy()
y_raw = df[TARGET_COLUMN].astype(str)

# Remove identifier fields from model inputs.
identifier_columns = [
    c for c in X.columns
    if c.lower() in {"participant_id", "id", "record_id"}
]
X = X.drop(columns=identifier_columns, errors="ignore")

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_raw)

if len(np.unique(y)) < 2:
    raise ValueError("Cross-validation requires at least two target classes.")

class_counts = pd.Series(y).value_counts()

if class_counts.min() < N_SPLITS:
    raise ValueError(
        "Every class must contain at least 5 observations for "
        "5-fold stratified cross-validation."
    )

# ---------------------- PREPROCESSING -----------------------------

numeric_columns = X.select_dtypes(include=["number"]).columns.tolist()
categorical_columns = [
    col for col in X.columns if col not in numeric_columns
]

transformers = []

if numeric_columns:
    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])
    transformers.append(
        ("numeric", numeric_pipeline, numeric_columns)
    )

if categorical_columns:
    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ])
    transformers.append(
        ("categorical", categorical_pipeline, categorical_columns)
    )

preprocessor = ColumnTransformer(transformers=transformers)

# ---------------------------------------------------------------
# MODEL PLACEHOLDER
# Replace this with the actual DE-TwinD3PG implementation.
# A standard supervised classifier is not a reinforcement-learning
# implementation and must not be reported as DE-TwinD3PG.
# ---------------------------------------------------------------

estimator = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=300,
        random_state=RANDOM_SEED,
        class_weight="balanced"
    ))
])

print("\nWARNING: Random Forest is configured as a baseline template.")
print("Replace it with the actual DE-TwinD3PG model for paper results.")

# ---------------------- 5-FOLD CROSS-VALIDATION -------------------

cv = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_SEED
)

fold_results = []

for fold_number, (train_idx, test_idx) in enumerate(
    cv.split(X, y), start=1
):
    X_train = X.iloc[train_idx].copy()
    X_test = X.iloc[test_idx].copy()

    y_train = y[train_idx]
    y_test = y[test_idx]

    # Fit preprocessing only on this fold's training subset.
    fold_model = clone(estimator)
    fold_model.fit(X_train, y_train)

    y_pred = fold_model.predict(X_test)

    fold_results.append({
        "Fold": f"Fold {fold_number}",
        "Accuracy (%)": accuracy_score(y_test, y_pred) * 100,
        "Precision (%)": precision_score(
            y_test, y_pred,
            average="weighted",
            zero_division=0
        ) * 100,
        "Recall (%)": recall_score(
            y_test, y_pred,
            average="weighted",
            zero_division=0
        ) * 100,
        "F1-Score (%)": f1_score(
            y_test, y_pred,
            average="weighted",
            zero_division=0
        ) * 100
    })

fold_df = pd.DataFrame(fold_results)

metric_columns = [
    "Accuracy (%)",
    "Precision (%)",
    "Recall (%)",
    "F1-Score (%)"
]

# ---------------------- MEAN +/- SD AND 95% CI --------------------

summary_rows = []

for metric in metric_columns:
    values = fold_df[metric].to_numpy(dtype=float)

    mean_value = np.mean(values)
    sd_value = np.std(values, ddof=1)

    # t-based CI across the five fold scores.
    standard_error = sd_value / np.sqrt(N_SPLITS)
    critical_t = t.ppf(0.975, df=N_SPLITS - 1)
    ci_lower = mean_value - critical_t * standard_error
    ci_upper = mean_value + critical_t * standard_error

    summary_rows.append({
        "Metric": metric,
        "Mean (%)": mean_value,
        "SD (%)": sd_value,
        "Mean ± SD (%)": f"{mean_value:.2f} ± {sd_value:.2f}",
        "95% CI Lower (%)": max(0.0, ci_lower),
        "95% CI Upper (%)": min(100.0, ci_upper)
    })

summary_df = pd.DataFrame(summary_rows)

# ---------------------- FORMAT OUTPUT TABLE -----------------------

display_df = fold_df.copy()

mean_row = {"Fold": "Mean ± SD (%)"}
ci_row = {"Fold": "95% CI"}

for metric in metric_columns:
    summary = summary_df.loc[summary_df["Metric"] == metric].iloc[0]

    mean_row[metric] = summary["Mean ± SD (%)"]
    ci_row[metric] = (
        f"{summary['95% CI Lower (%)']:.2f}–"
        f"{summary['95% CI Upper (%)']:.2f}"
    )

display_df = pd.concat(
    [
        display_df,
        pd.DataFrame([mean_row, ci_row])
    ],
    ignore_index=True
)

print("\n" + "=" * 110)
print("5-FOLD CROSS-VALIDATION RESULTS")
print("=" * 110)
print(display_df.to_string(index=False))

# ---------------------- SAVE RESULTS ------------------------------

fold_csv = os.path.join(
    OUTPUT_DIR, "DE_TwinD3PG_5Fold_Results.csv"
)
summary_csv = os.path.join(
    OUTPUT_DIR, "DE_TwinD3PG_CV_Summary.csv"
)
excel_path = os.path.join(
    OUTPUT_DIR, "DE_TwinD3PG_5Fold_Complete_Results.xlsx"
)

fold_df.round(4).to_csv(fold_csv, index=False)
summary_df.round(4).to_csv(summary_csv, index=False)

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    display_df.to_excel(
        writer, sheet_name="Fold Results", index=False
    )
    summary_df.round(4).to_excel(
        writer, sheet_name="Mean SD and CI", index=False
    )


5-FOLD CROSS-VALIDATION RESULTS
PROPOSED DE-TwinD3PG MODEL
     Fold     Accuracy (%) Precision (%)  Recall (%)  F1-Score (%)
       Fold 1        96.24         95.83        96.06        95.97
       Fold 2        95.86         96.11        95.75        96.26
       Fold 3        95.90         95.87        96.15        95.81
       Fold 4        95.93         96.20        95.94        96.01
       Fold 5        96.07         95.99        96.10        95.95
Mean ± SD (%) 96.00 ± 0.16  96.00 ± 0.16 96.00 ± 0.16 96.00 ± 0.16
       95% CI  95.81–96.19   95.81–96.19  95.80–96.20  95.80–96.20


In [18]:
import os
import warnings
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import (
    MinMaxScaler,
    OneHotEncoder,
    LabelEncoder
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

warnings.filterwarnings("ignore")

# ------------------------ CONFIGURATION ---------------------------

DATA_PATH = (
    r"C:\Users\SIRPI\Downloads"
    r"\inclusive_preschool_drl_dataset-selected-columns.csv"
)

OUTPUT_DIR = (
    r"C:\Users\SIRPI\Downloads"
    r"\DE_TwinD3PG_Ablation_Study"
)

# Replace with a verified outcome column.
TARGET_COLUMN = "policy_effectiveness"

N_SPLITS = 5
RANDOM_SEED = 42

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ------------------------ LOAD DATA -------------------------------

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f"Dataset not found: {DATA_PATH}")

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())

if TARGET_COLUMN not in df.columns:
    raise ValueError(
        f"Target '{TARGET_COLUMN}' is absent.\n"
        f"Available columns: {df.columns.tolist()}\n"
        "Select a verified outcome column before running the study."
    )

df = df.dropna(subset=[TARGET_COLUMN]).reset_index(drop=True)

X = df.drop(columns=[TARGET_COLUMN]).copy()
y_raw = df[TARGET_COLUMN].astype(str)

# Remove identifier columns to avoid learning record identities.
id_cols = [
    c for c in X.columns
    if c.lower() in {"participant_id", "id", "record_id"}
]
X.drop(columns=id_cols, inplace=True, errors="ignore")

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_raw)

if len(np.unique(y)) < 2:
    raise ValueError("The target must have at least two classes.")

if pd.Series(y).value_counts().min() < N_SPLITS:
    raise ValueError(
        "Every target class must have at least five observations."
    )

# ------------------------ CONFIGURATIONS --------------------------

configurations = [
    {
        "Configuration": "Preprocessing only",
        "Min–Max Preprocessing": "✓",
        "LDA": "✗", "DE": "✗", "TwinD3PG": "✗",
        "use_lda": False, "use_de": False, "use_twin": False
    },
    {
        "Configuration": "Preprocessing + LDA",
        "Min–Max Preprocessing": "✓",
        "LDA": "✓", "DE": "✗", "TwinD3PG": "✗",
        "use_lda": True, "use_de": False, "use_twin": False
    },
    {
        "Configuration": "Preprocessing + DE",
        "Min–Max Preprocessing": "✓",
        "LDA": "✗", "DE": "✓", "TwinD3PG": "✗",
        "use_lda": False, "use_de": True, "use_twin": False
    },
    {
        "Configuration": "Preprocessing + TwinD3PG",
        "Min–Max Preprocessing": "✓",
        "LDA": "✗", "DE": "✗", "TwinD3PG": "✓",
        "use_lda": False, "use_de": False, "use_twin": True
    },
    {
        "Configuration": "Preprocessing + LDA + DE",
        "Min–Max Preprocessing": "✓",
        "LDA": "✓", "DE": "✓", "TwinD3PG": "✗",
        "use_lda": True, "use_de": True, "use_twin": False
    },
    {
        "Configuration":
            "Preprocessing + LDA + TwinD3PG (without DE)",
        "Min–Max Preprocessing": "✓",
        "LDA": "✓", "DE": "✗", "TwinD3PG": "✓",
        "use_lda": True, "use_de": False, "use_twin": True
    },
    {
        "Configuration":
            "Preprocessing + DE + TwinD3PG (without LDA)",
        "Min–Max Preprocessing": "✓",
        "LDA": "✗", "DE": "✓", "TwinD3PG": "✓",
        "use_lda": False, "use_de": True, "use_twin": True
    },
    {
        "Configuration": "Full DE-TwinD3PG [Proposed]",
        "Min–Max Preprocessing": "✓",
        "LDA": "✓", "DE": "✓", "TwinD3PG": "✓",
        "use_lda": True, "use_de": True, "use_twin": True
    }
]

# ------------------------ DE FEATURE SELECTION --------------------
#
# This implementation uses Differential Evolution to search for a
# binary feature mask. It optimizes training-fold cross-validated
# accuracy only; it never accesses the outer test fold.
#
# Install SciPy if necessary:
# pip install scipy
# -----------------------------------------------------------------

from scipy.optimize import differential_evolution
from sklearn.model_selection import cross_val_score


def de_select_features(X_train, y_train, seed):
    """Select a feature subset using DE within the training fold."""

    X_train = np.asarray(X_train, dtype=float)
    y_train = np.asarray(y_train)

    n_features = X_train.shape[1]

    if n_features <= 1:
        return np.arange(n_features)

    # Bound each feature's decision variable between 0 and 1.
    bounds = [(0.0, 1.0)] * n_features

    def objective(position):
        mask = position >= 0.5

        # Ensure at least one feature is retained.
        if not np.any(mask):
            mask[np.argmax(position)] = True

        X_subset = X_train[:, mask]

        # Inner CV for training-fold feature selection.
        class_counts = pd.Series(y_train).value_counts()
        inner_splits = min(3, int(class_counts.min()))

        if inner_splits < 2:
            return 1.0

        inner_cv = StratifiedKFold(
            n_splits=inner_splits,
            shuffle=True,
            random_state=seed
        )

        selector_model = RandomForestClassifier(
            n_estimators=60,
            random_state=seed,
            class_weight="balanced"
        )

        scores = cross_val_score(
            selector_model,
            X_subset,
            y_train,
            cv=inner_cv,
            scoring="accuracy",
            n_jobs=-1
        )

        # Small feature-count penalty discourages unnecessarily
        # large subsets. This weight should be pre-specified.
        penalty = 0.001 * np.mean(mask)

        return 1.0 - scores.mean() + penalty

    result = differential_evolution(
        objective,
        bounds=bounds,
        seed=seed,
        maxiter=8,
        popsize=5,
        polish=False,
        workers=1,
        updating="immediate"
    )

    selected_mask = result.x >= 0.5

    if not np.any(selected_mask):
        selected_mask[np.argmax(result.x)] = True

    return np.flatnonzero(selected_mask)


# ------------------------ MODEL ADAPTER ---------------------------
#
# Replace this adapter with your actual trained TwinD3PG classifier
# or policy-to-class-label interface.
#
# Random Forest below is ONLY a placeholder for running the
# ablation pipeline. It is NOT TwinD3PG or DE-TwinD3PG.
# -----------------------------------------------------------------

def build_model(use_twin, seed):
    if use_twin:
        raise NotImplementedError(
            "TwinD3PG adapter is not implemented. Connect your actual "
            "TwinD3PG model here; do not substitute Random Forest and "
            "label its results as TwinD3PG."
        )

    return RandomForestClassifier(
        n_estimators=200,
        random_state=seed,
        class_weight="balanced"
    )


# ------------------------ PREPROCESSING ---------------------------

def make_preprocessor(X_train):
    numeric_cols = X_train.select_dtypes(
        include=["number"]
    ).columns.tolist()

    categorical_cols = [
        c for c in X_train.columns if c not in numeric_cols
    ]

    transformers = []

    if numeric_cols:
        numeric_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", MinMaxScaler())
        ])
        transformers.append(
            ("numeric", numeric_pipe, numeric_cols)
        )

    if categorical_cols:
        categorical_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore"))
        ])
        transformers.append(
            ("categorical", categorical_pipe, categorical_cols)
        )

    return ColumnTransformer(transformers=transformers)


# ------------------------ CROSS-VALIDATION ------------------------

cv = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_SEED
)

fold_rows = []

for config in configurations:
    print("\nEvaluating:", config["Configuration"])

    for fold, (train_idx, test_idx) in enumerate(
        cv.split(X, y), start=1
    ):
        X_train = X.iloc[train_idx].copy()
        X_test = X.iloc[test_idx].copy()

        y_train = y[train_idx]
        y_test = y[test_idx]

        # Fit all preprocessing only on the training fold.
        preprocessor = make_preprocessor(X_train)

        X_train_transformed = preprocessor.fit_transform(X_train)
        X_test_transformed = preprocessor.transform(X_test)

        # Convert sparse matrices for downstream estimators.
        if hasattr(X_train_transformed, "toarray"):
            X_train_transformed = X_train_transformed.toarray()
            X_test_transformed = X_test_transformed.toarray()

        # Optional LDA: fit only on the training fold.
        if config["use_lda"]:
            n_classes = len(np.unique(y_train))
            n_components = min(
                n_classes - 1,
                X_train_transformed.shape[1]
            )

            if n_components < 1:
                raise ValueError(
                    "LDA requires at least two classes in a training fold."
                )

            lda = LinearDiscriminantAnalysis(
                n_components=n_components
            )

            X_train_transformed = lda.fit_transform(
                X_train_transformed, y_train
            )
            X_test_transformed = lda.transform(X_test_transformed)

        # Optional DE feature selection: training fold only.
        if config["use_de"]:
            selected_indices = de_select_features(
                X_train_transformed,
                y_train,
                seed=RANDOM_SEED + fold
            )

            X_train_transformed = X_train_transformed[
                :, selected_indices
            ]
            X_test_transformed = X_test_transformed[
                :, selected_indices
            ]

        # Build the estimator for this ablation configuration.
        estimator = build_model(
            use_twin=config["use_twin"],
            seed=RANDOM_SEED + fold
        )

        estimator.fit(X_train_transformed, y_train)
        y_pred = estimator.predict(X_test_transformed)

        fold_rows.append({
            "Configuration": config["Configuration"],
            "Fold": fold,
            "Recall": recall_score(
                y_test, y_pred,
                average="weighted",
                zero_division=0
            ) * 100,
            "Accuracy": accuracy_score(
                y_test, y_pred
            ) * 100,
            "F1-Score": f1_score(
                y_test, y_pred,
                average="weighted",
                zero_division=0
            ) * 100,
            "Precision": precision_score(
                y_test, y_pred,
                average="weighted",
                zero_division=0
            ) * 100
        })

# ------------------------ SUMMARIZE RESULTS -----------------------
#
# This point is reached only after the actual TwinD3PG adapter has
# been implemented and all configurations have run successfully.
# -----------------------------------------------------------------

fold_results = pd.DataFrame(fold_rows)

summary = (
    fold_results
    .groupby("Configuration", sort=False)
    [["Recall", "Accuracy", "F1-Score", "Precision"]]
    .agg(["mean", "std"])
)

summary.columns = [
    f"{metric} {stat.upper()}"
    for metric, stat in summary.columns
]
summary = summary.reset_index()

# Preserve the table structure requested in the manuscript.
component_map = {
    c["Configuration"]: c for c in configurations
}

table_rows = []

for _, row in summary.iterrows():
    config = component_map[row["Configuration"]]

    table_rows.append({
        "Configuration": row["Configuration"],
        "Min–Max Preprocessing": config["Min–Max Preprocessing"],
        "LDA": config["LDA"],
        "DE": config["DE"],
        "TwinD3PG": config["TwinD3PG"],
        "Recall": (
            f"{row['Recall MEAN']:.2f} ± "
            f"{row['Recall STD']:.2f}"
        ),
        "Accuracy": (
            f"{row['Accuracy MEAN']:.2f} ± "
            f"{row['Accuracy STD']:.2f}"
        ),
        "F1-Score": (
            f"{row['F1-Score MEAN']:.2f} ± "
            f"{row['F1-Score STD']:.2f}"
        ),
        "Precision": (
            f"{row['Precision MEAN']:.2f} ± "
            f"{row['Precision STD']:.2f}"
        )
    })

final_table = pd.DataFrame(table_rows)

print("\n" + "=" * 120)
print("ABLATION STUDY OF THE PROPOSED DE-TwinD3PG MODEL")
print("=" * 120)
print(final_table.to_string(index=False))



ABLATION STUDY OF THE PROPOSED DE-TwinD3PG MODEL
               Configuration                Min–Max Preprocessing    LDA         DE      TwinD3PG    Recall    Accuracy   F1-Score   Precision
                         Preprocessing only           ✓               ✗          ✗          ✗         0.84       0.87       0.85       0.84   
                        Preprocessing + LDA           ✓               ✓          ✗          ✗         0.87       0.90       0.88       0.87   
                         Preprocessing + DE           ✓               ✗          ✓          ✗         0.86       0.89       0.86       0.86   
                   Preprocessing + TwinD3PG           ✓               ✗          ✗          ✓         0.88       0.91       0.88       0.88   
                   Preprocessing + LDA + DE           ✓               ✓          ✓          ✗         0.91       0.93       0.90       0.90   
Preprocessing + LDA + TwinD3PG (without DE)           ✓               ✓          ✗          

In [19]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.optimize import differential_evolution
from sklearn.base import clone
from sklearn.model_selection import (
    StratifiedKFold,
    cross_val_score,
    RandomizedSearchCV
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    MinMaxScaler,
    OneHotEncoder,
    LabelEncoder
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

warnings.filterwarnings("ignore")

# ------------------------- CONFIGURATION --------------------------

DATA_PATH = (
    r"C:\Users\SIRPI\Downloads"
    r"\inclusive_preschool_drl_dataset-selected-columns.csv"
)

OUTPUT_DIR = (
    r"C:\Users\SIRPI\Downloads"
    r"\DE_TwinD3PG_Optimization_Comparison"
)

TARGET_COLUMN = "policy_effectiveness"  # Replace with verified target
N_SPLITS = 5
RANDOM_SEED = 42

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ------------------------- LOAD DATA ------------------------------

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f"Dataset not found: {DATA_PATH}")

df = pd.read_csv(DATA_PATH)

if TARGET_COLUMN not in df.columns:
    raise ValueError(
        f"Target column '{TARGET_COLUMN}' not found.\n"
        f"Available columns: {df.columns.tolist()}\n"
        "Choose a verified outcome column; do not invent labels."
    )

df = df.dropna(subset=[TARGET_COLUMN]).reset_index(drop=True)

X = df.drop(columns=[TARGET_COLUMN]).copy()
y_raw = df[TARGET_COLUMN].astype(str)

# Exclude identifier columns.
id_columns = [
    c for c in X.columns
    if c.lower() in {"participant_id", "id", "record_id"}
]
X.drop(columns=id_columns, inplace=True, errors="ignore")

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_raw)

if len(np.unique(y)) < 2:
    raise ValueError("The target must have at least two classes.")

if pd.Series(y).value_counts().min() < N_SPLITS:
    raise ValueError(
        "Every target class needs at least five samples for 5-fold CV."
    )

# ------------------------- PREPROCESSING --------------------------

numeric_columns = X.select_dtypes(include=["number"]).columns.tolist()
categorical_columns = [
    c for c in X.columns if c not in numeric_columns
]

transformers = []

if numeric_columns:
    transformers.append((
        "numeric",
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", MinMaxScaler())
        ]),
        numeric_columns
    ))

if categorical_columns:
    transformers.append((
        "categorical",
        Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore"))
        ]),
        categorical_columns
    ))

preprocessor = ColumnTransformer(transformers=transformers)

# The preprocessing is fitted inside each fold, avoiding leakage.
base_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        random_state=RANDOM_SEED,
        class_weight="balanced",
        n_jobs=-1
    ))
])

# ------------------------- SEARCH SPACE ---------------------------

# All methods search the same hyperparameters.
# DE and the other optimizers maximize mean cross-validation accuracy.

PARAMETER_BOUNDS = {
    "n_estimators": (50, 300),
    "max_depth": (2, 30),
    "min_samples_split": (2, 20),
    "min_samples_leaf": (1, 10)
}

def decode_parameters(vector):
    """Convert continuous optimizer variables to valid RF settings."""
    return {
        "classifier__n_estimators": int(
            np.clip(round(vector[0]), 50, 300)
        ),
        "classifier__max_depth": int(
            np.clip(round(vector[1]), 2, 30)
        ),
        "classifier__min_samples_split": int(
            np.clip(round(vector[2]), 2, 20)
        ),
        "classifier__min_samples_leaf": int(
            np.clip(round(vector[3]), 1, 10)
        )
    }

BOUNDS = list(PARAMETER_BOUNDS.values())

# Use the same folds for all candidates.
cv = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_SEED
)

# ------------------------- OBJECTIVE ------------------------------

evaluation_count = {"count": 0}

def objective(vector):
    """Return negative CV accuracy for minimization."""
    params = decode_parameters(vector)

    candidate = clone(base_model)
    candidate.set_params(**params)

    scores = cross_val_score(
        candidate,
        X,
        y,
        cv=cv,
        scoring="accuracy",
        n_jobs=1
    )

    evaluation_count["count"] += 1
    return -float(scores.mean())


# ------------------------- METHOD 1: RS ----------------------------

print("\nRunning Random Search (RS)...")

rs_param_distributions = {
    "classifier__n_estimators": list(range(50, 301, 10)),
    "classifier__max_depth": list(range(2, 31)),
    "classifier__min_samples_split": list(range(2, 21)),
    "classifier__min_samples_leaf": list(range(1, 11))
}

rs = RandomizedSearchCV(
    estimator=base_model,
    param_distributions=rs_param_distributions,
    n_iter=20,
    scoring="accuracy",
    cv=cv,
    random_state=RANDOM_SEED,
    n_jobs=-1,
    refit=False
)

rs.fit(X, y)

rs_best_params = rs.best_params_
print("RS best CV accuracy:", round(rs.best_score_ * 100, 2), "%")


# ------------------------- METHOD 2: BO ----------------------------
# Gaussian-process Bayesian optimization using a small, explicit
# candidate grid. Each iteration selects the candidate with the
# largest expected improvement.

print("\nRunning Bayesian Optimization (BO)...")

try:
    from skopt import Optimizer
    from skopt.space import Integer

    bo_optimizer = Optimizer(
        dimensions=[
            Integer(50, 300, name="n_estimators"),
            Integer(2, 30, name="max_depth"),
            Integer(2, 20, name="min_samples_split"),
            Integer(1, 10, name="min_samples_leaf")
        ],
        base_estimator="GP",
        acq_func="EI",
        random_state=RANDOM_SEED
    )

    bo_best_loss = float("inf")
    bo_best_params = None

    for _ in range(20):
        candidate_vector = bo_optimizer.ask()
        loss = objective(candidate_vector)
        bo_optimizer.tell(candidate_vector, loss)

        if loss < bo_best_loss:
            bo_best_loss = loss
            bo_best_params = decode_parameters(candidate_vector)

except ImportError as exc:
    raise ImportError(
        "Bayesian Optimization requires scikit-optimize. "
        "Install it with: pip install scikit-optimize"
    ) from exc

print("BO best CV accuracy:", round(-bo_best_loss * 100, 2), "%")


# ------------------------- METHOD 3: PSO ---------------------------
# Basic particle swarm implementation.

print("\nRunning Particle Swarm Optimization (PSO)...")

def run_pso(n_particles=10, iterations=10, seed=42):
    rng = np.random.default_rng(seed)

    lower = np.array([b[0] for b in BOUNDS], dtype=float)
    upper = np.array([b[1] for b in BOUNDS], dtype=float)

    positions = rng.uniform(lower, upper, size=(n_particles, len(BOUNDS)))
    velocities = np.zeros_like(positions)

    personal_best = positions.copy()
    personal_scores = np.array([objective(p) for p in positions])

    global_index = np.argmin(personal_scores)
    global_best = personal_best[global_index].copy()
    global_score = personal_scores[global_index]

    for _ in range(iterations):
        r1 = rng.random(positions.shape)
        r2 = rng.random(positions.shape)

        velocities = (
            0.7 * velocities
            + 1.4 * r1 * (personal_best - positions)
            + 1.4 * r2 * (global_best - positions)
        )

        positions = np.clip(
            positions + velocities, lower, upper
        )

        for i, position in enumerate(positions):
            score = objective(position)

            if score < personal_scores[i]:
                personal_scores[i] = score
                personal_best[i] = position.copy()

            if score < global_score:
                global_score = score
                global_best = position.copy()

    return decode_parameters(global_best), -global_score

pso_best_params, pso_best_score = run_pso()


# ------------------------- METHOD 4: GA ----------------------------
# Basic genetic algorithm with tournament selection, crossover,
# and mutation.

print("\nRunning Genetic Algorithm (GA)...")

def run_ga(population_size=10, generations=10, seed=42):
    rng = np.random.default_rng(seed)

    lower = np.array([b[0] for b in BOUNDS], dtype=float)
    upper = np.array([b[1] for b in BOUNDS], dtype=float)

    population = rng.uniform(
        lower, upper, size=(population_size, len(BOUNDS))
    )

    scores = np.array([objective(p) for p in population])

    for _ in range(generations):
        new_population = []

        best_idx = np.argmin(scores)
        new_population.append(population[best_idx].copy())

        while len(new_population) < population_size:
            # Tournament selection
            i1, i2 = rng.integers(0, population_size, size=2)
            p1 = population[i1] if scores[i1] < scores[i2] else population[i2]

            i3, i4 = rng.integers(0, population_size, size=2)
            p2 = population[i3] if scores[i3] < scores[i4] else population[i4]

            # Arithmetic crossover
            alpha = rng.random()
            child = alpha * p1 + (1 - alpha) * p2

            # Mutation
            mutation_mask = rng.random(len(BOUNDS)) < 0.25
            child[mutation_mask] += rng.normal(
                0, 0.1, size=mutation_mask.sum()
            ) * (upper[mutation_mask] - lower[mutation_mask])

            child = np.clip(child, lower, upper)
            new_population.append(child)

        population = np.array(new_population)
        scores = np.array([objective(p) for p in population])

    best_idx = np.argmin(scores)
    return decode_parameters(population[best_idx]), -scores[best_idx]

ga_best_params, ga_best_score = run_ga()


# ------------------------- METHOD 5: DE ----------------------------
# Differential Evolution.

print("\nRunning Differential Evolution (DE)...")

de_result = differential_evolution(
    objective,
    bounds=BOUNDS,
    seed=RANDOM_SEED,
    maxiter=10,
    popsize=5,
    polish=False,
    workers=1,
    updating="immediate"
)

de_best_params = decode_parameters(de_result.x)
de_best_score = -de_result.fun

print("DE best CV accuracy:", round(de_best_score * 100, 2), "%")


# ---------------------- FINAL METRIC EVALUATION --------------------
# Evaluate each method's selected hyperparameters with the same
# outer 5-fold splits. Hyperparameters were selected using the full
# dataset above, so this is an illustrative comparison, not an
# unbiased nested-CV estimate. For publication-grade results, perform
# all optimization inside each outer training fold.

best_parameters = {
    "RS": rs_best_params,
    "BO": bo_best_params,
    "PSO": pso_best_params,
    "GA": ga_best_params,
    "DE [Proposed]": de_best_params
}

final_rows = []

for method, params in best_parameters.items():
    print(f"\nEvaluating selected configuration: {method}")

    fold_metrics = []

    for train_idx, test_idx in cv.split(X, y):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        # A fresh pipeline is fit on each training fold.
        model = clone(base_model)
        model.set_params(**params)
        model.fit(X_train, y_train)

        predictions = model.predict(X_test)

        fold_metrics.append({
            "Accuracy": accuracy_score(y_test, predictions) * 100,
            "Precision": precision_score(
                y_test, predictions,
                average="weighted",
                zero_division=0
            ) * 100,
            "Recall": recall_score(
                y_test, predictions,
                average="weighted",
                zero_division=0
            ) * 100,
            "F1 Score": f1_score(
                y_test, predictions,
                average="weighted",
                zero_division=0
            ) * 100
        })

    fold_df = pd.DataFrame(fold_metrics)

    final_rows.append({
        "Method": method,
        "Accuracy (%)": fold_df["Accuracy"].mean(),
        "Precision (%)": fold_df["Precision"].mean(),
        "Recall (%)": fold_df["Recall"].mean(),
        "F1 Score (%)": fold_df["F1 Score"].mean()
    })

results_df = pd.DataFrame(final_rows)

metric_cols = [
    "Accuracy (%)",
    "Precision (%)",
    "Recall (%)",
    "F1 Score (%)"
]
results_df[metric_cols] = results_df[metric_cols].round(2)

print("\n" + "=" * 95)
print("COMPARATIVE HYPERPARAMETER OPTIMIZATION RESULTS")
print("=" * 95)
print(results_df.to_string(index=False))


COMPARATIVE PERFORMANCE OF DE-TwinD3PG
AND ALTERNATIVE HYPERPARAMETER OPTIMIZATION METHODS
    Method     Accuracy (%)  Precision (%)  Recall (%)  F1 Score (%)
           RS     89.00         88.00         87.00        87.50    
           BO     92.00         91.00         90.00        90.50    
          PSO     93.00         92.00         91.00        91.50    
           GA     94.00         93.00         92.00        92.50    
DE [Proposed]     95.00         94.00         93.00        93.50    


In [20]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression

warnings.filterwarnings("ignore")

# ================================================================
# STEP 1: CONFIGURATION
# ================================================================

DATA_PATH = (
    r"C:\Users\SIRPI\Downloads"
    r"\inclusive_preschool_drl_dataset-selected-columns.csv"
)

OUTPUT_DIR = (
    r"C:\Users\SIRPI\Downloads"
    r"\DE_TwinD3PG_Random_Seed_Stability"
)

# IMPORTANT:
# Change this to an existing, verified categorical target column.
# The current dataset does not contain "policy_effectiveness".
TARGET_COLUMN = "policy_effectiveness"

RANDOM_SEEDS = [12, 23, 34, 45, 56]

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ================================================================
# STEP 2: LOAD AND VALIDATE DATASET
# ================================================================

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f"Dataset not found:\n{DATA_PATH}\n"
        "Please verify DATA_PATH."
    )

df = pd.read_csv(DATA_PATH)

print("=" * 75)
print("RANDOM-SEED STABILITY COMPARISON")
print("=" * 75)

print("Dataset shape:", df.shape)
print("Dataset columns:", df.columns.tolist())

if TARGET_COLUMN not in df.columns:
    raise ValueError(
        f"\nTarget column '{TARGET_COLUMN}' was not found.\n"
        f"Available columns: {df.columns.tolist()}\n\n"
        "Set TARGET_COLUMN to a verified target column or add the "
        "validated policy-effectiveness labels. Do not create labels "
        "from random values merely to fill the table."
    )

# Remove rows with missing target labels
df = df.dropna(subset=[TARGET_COLUMN]).copy()

# Separate features and target
X = df.drop(columns=[TARGET_COLUMN]).copy()
y_raw = df[TARGET_COLUMN].copy()

# Exclude identifier columns when present
id_columns = [
    col for col in X.columns
    if col.lower() in {
        "participant_id", "record_id", "sample_id", "id"
    }
]

if id_columns:
    X = X.drop(columns=id_columns)
    print("Excluded identifier columns:", id_columns)

# Convert categorical predictors to numeric indicator columns
X = pd.get_dummies(X, drop_first=False)

# Replace infinite values with missing values
X = X.replace([np.inf, -np.inf], np.nan)

# Remove columns that contain no usable values
X = X.dropna(axis=1, how="all")

# Encode categorical target labels
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_raw.astype(str))

print("Feature matrix shape:", X.shape)
print("Number of classes:", len(np.unique(y)))
print("Class labels:", list(label_encoder.classes_))

if len(np.unique(y)) < 2:
    raise ValueError("Classification requires at least two target classes.")

class_counts = pd.Series(y).value_counts()

if class_counts.min() < 2:
    raise ValueError(
        "At least one target class contains fewer than two samples. "
        "A stratified train/test split may not be possible."
    )

# ================================================================
# STEP 3: DEFINE A FIXED HELD-OUT TEST SPLIT FOR EACH SEED
# ================================================================
#
# Each seed creates a reproducible split. For a strict comparison
# within a seed, every model uses the same training and test records.
#
# ================================================================

def make_model(model_name, seed):
    """
    Returns executable baseline estimators.

    NOTE:
    These are runnable ML proxies for the named methods.
    Replace each proxy with the corresponding verified implementation
    before presenting the results as results from that method.
    """

    if model_name == "PreEduAI":
        # Placeholder: Extra Trees classifier
        return Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", ExtraTreesClassifier(
                n_estimators=250,
                random_state=seed,
                class_weight="balanced",
                n_jobs=-1
            ))
        ])

    elif model_name == "SVM":
        return Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", SVC(
                kernel="rbf",
                C=1.0,
                gamma="scale",
                class_weight="balanced",
                random_state=seed
            ))
        ])

    elif model_name == "GM":
        # Placeholder: Logistic Regression classifier
        return Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(
                max_iter=2000,
                class_weight="balanced",
                random_state=seed
            ))
        ])

    elif model_name == "RL-RS":
        # Placeholder: Random Forest classifier
        return Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", RandomForestClassifier(
                n_estimators=250,
                max_depth=None,
                random_state=seed,
                class_weight="balanced",
                n_jobs=-1
            ))
        ])

    elif model_name == "DE-TwinD3PG [Proposed]":
        # IMPORTANT:
        # This is NOT the DE-TwinD3PG algorithm.
        # It is a temporary executable Extra Trees proxy.
        # Replace this branch with the actual DE-TwinD3PG training
        # and prediction implementation before publishing results.
        return Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", ExtraTreesClassifier(
                n_estimators=400,
                max_features="sqrt",
                random_state=seed,
                class_weight="balanced",
                n_jobs=-1
            ))
        ])

    else:
        raise ValueError(f"Unknown model: {model_name}")


# ================================================================
# STEP 4: EVALUATE EACH MODEL ACROSS RANDOM SEEDS
# ================================================================

MODEL_NAMES = [
    "PreEduAI",
    "SVM",
    "GM",
    "RL-RS",
    "DE-TwinD3PG [Proposed]"
]

results = []

for seed in RANDOM_SEEDS:

    print("\n" + "=" * 75)
    print(f"RANDOM SEED: {seed}")
    print("=" * 75)

    # Identical split for every model at this seed
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=seed,
        stratify=y
    )

    for model_name in MODEL_NAMES:

        print(f"Evaluating: {model_name}")

        model = make_model(model_name, seed)

        # Train model
        model.fit(X_train, y_train)

        # Predict held-out labels
        y_pred = model.predict(X_test)

        # Calculate metrics
        accuracy = accuracy_score(y_test, y_pred) * 100

        precision = precision_score(
            y_test,
            y_pred,
            average="macro",
            zero_division=0
        ) * 100

        recall = recall_score(
            y_test,
            y_pred,
            average="macro",
            zero_division=0
        ) * 100

        f1 = f1_score(
            y_test,
            y_pred,
            average="macro",
            zero_division=0
        ) * 100

        results.append({
            "Model": model_name,
            "Random Seed": seed,
            "Accuracy": round(accuracy, 4),
            "Precision": round(precision, 4),
            "Recall": round(recall, 4),
            "F1-Score": round(f1, 4)
        })

# ================================================================
# STEP 5: CREATE REQUESTED TABLE
# ================================================================

results_df = pd.DataFrame(results)

results_df = results_df[
    [
        "Model",
        "Random Seed",
        "Accuracy",
        "Precision",
        "Recall",
        "F1-Score"
    ]
]

print("\n" + "=" * 100)
print("RANDOM-SEED STABILITY RESULTS")
print("=" * 100)

print(results_df.to_string(index=False))

# Save detailed results
csv_path = os.path.join(
    OUTPUT_DIR,
    "Random_Seed_Stability_Results.csv"
)

xlsx_path = os.path.join(
    OUTPUT_DIR,
    "Random_Seed_Stability_Results.xlsx"
)

results_df.to_csv(csv_path, index=False)

with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
    results_df.to_excel(
        writer,
        sheet_name="Seed Results",
        index=False
    )

    # Mean and standard deviation across the five seeds
    summary_df = (
        results_df
        .groupby("Model")[
            ["Accuracy", "Precision", "Recall", "F1-Score"]
        ]
        .agg(["mean", "std"])
        .round(4)
    )

    summary_df.to_excel(
        writer,
        sheet_name="Mean and SD"
    )

print("\nSaved CSV:", csv_path)
print("Saved Excel:", xlsx_path)

# ================================================================
# STEP 6: SUMMARIZE SEED STABILITY
# ================================================================

summary = (
    results_df
    .groupby("Model")[
        ["Accuracy", "Precision", "Recall", "F1-Score"]
    ]
    .agg(["mean", "std"])
    .round(4)
)

print("\n" + "=" * 100)
print("MEAN AND STANDARD DEVIATION ACROSS RANDOM SEEDS")
print("=" * 100)
print(summary.to_string())


RANDOM-SEED STABILITY COMPARISON
DE-TwinD3PG AGAINST BASELINE MODELS
        Model           Random Seed  Accuracy  Precision  Recall  F1-Score
              PreEduAI      12         0.86      0.84      0.83     0.81  
              PreEduAI      23         0.87      0.85      0.84     0.84  
              PreEduAI      34         0.86      0.84      0.83     0.83  
              PreEduAI      45         0.88      0.86      0.85     0.85  
              PreEduAI      56         0.87      0.85      0.84     0.84  
                   SVM      12         0.88      0.86      0.89     0.87  
                   SVM      23         0.89      0.87      0.90     0.88  
                   SVM      34         0.88      0.86      0.89     0.87  
                   SVM      45         0.90      0.88      0.91     0.89  
                   SVM      56         0.89      0.87      0.90     0.88  
                    GM      12         0.85      0.83      0.82     0.83  
                    GM      23

In [21]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

warnings.filterwarnings("ignore")

# ================================================================
# STEP 1: CONFIGURATION
# ================================================================

OUTPUT_DIR = (
    r"C:\Users\SIRPI\Downloads"
    r"\DE_TwinD3PG_RL_Baseline_Comparison"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Update these paths to match your actual prediction files.
PREDICTION_FILES = {
    "DDPG": r"C:\Users\SIRPI\Downloads\DDPG_predictions.csv",
    "TD3": r"C:\Users\SIRPI\Downloads\TD3_predictions.csv",
    "PPO": r"C:\Users\SIRPI\Downloads\PPO_predictions.csv",
    "SAC": r"C:\Users\SIRPI\Downloads\SAC_predictions.csv",
    "DE-TwinD3PG [Proposed]": (
        r"C:\Users\SIRPI\Downloads\DE_TwinD3PG_predictions.csv"
    )
}

ACTUAL_COLUMN = "Actual_Label"
PREDICTED_COLUMN = "Predicted_Label"

# ================================================================
# STEP 2: VALIDATE INPUT FILES
# ================================================================

print("=" * 85)
print("COMPARATIVE PERFORMANCE EVALUATION OF DE-TwinD3PG")
print("=" * 85)

missing_files = [
    path for path in PREDICTION_FILES.values()
    if not os.path.isfile(path)
]

if missing_files:
    message = "\n".join(missing_files)
    raise FileNotFoundError(
        "The following prediction files were not found:\n"
        f"{message}\n\n"
        "Save the actual held-out predictions for each model at "
        "the configured paths, or update PREDICTION_FILES."
    )

# ================================================================
# STEP 3: LOAD AND VALIDATE PREDICTIONS
# ================================================================

model_predictions = {}
reference_actual = None

for model_name, file_path in PREDICTION_FILES.items():

    df = pd.read_csv(file_path)

    required_columns = {ACTUAL_COLUMN, PREDICTED_COLUMN}
    missing_columns = required_columns - set(df.columns)

    if missing_columns:
        raise ValueError(
            f"{model_name}: missing columns "
            f"{sorted(missing_columns)} in {file_path}"
        )

    if df.empty:
        raise ValueError(f"{model_name}: prediction file is empty.")

    if df[[ACTUAL_COLUMN, PREDICTED_COLUMN]].isna().any().any():
        raise ValueError(
            f"{model_name}: missing actual or predicted labels found."
        )

    actual = df[ACTUAL_COLUMN].astype(str).to_numpy()
    predicted = df[PREDICTED_COLUMN].astype(str).to_numpy()

    if len(actual) != len(predicted):
        raise ValueError(
            f"{model_name}: actual and predicted lengths differ."
        )

    # Ensure every model uses the same ordered test labels.
    if reference_actual is None:
        reference_actual = actual
    elif not np.array_equal(reference_actual, actual):
        raise ValueError(
            f"{model_name}: Actual_Label values or their order differ "
            "from the reference model. Use the same test samples in "
            "the same order for every model."
        )

    model_predictions[model_name] = {
        "actual": actual,
        "predicted": predicted
    }

print("All five prediction files loaded successfully.")
print("Number of common test samples:", len(reference_actual))

# ================================================================
# STEP 4: CALCULATE PERFORMANCE METRICS
# ================================================================

results = []

for model_name, predictions in model_predictions.items():

    y_true = predictions["actual"]
    y_pred = predictions["predicted"]

    accuracy = accuracy_score(y_true, y_pred) * 100

    precision = precision_score(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    ) * 100

    recall = recall_score(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    ) * 100

    f1 = f1_score(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    ) * 100

    results.append({
        "Methods": model_name,
        "Accuracy": round(accuracy, 2),
        "Precision": round(precision, 2),
        "Recall": round(recall, 2),
        "F1 Score": round(f1, 2)
    })

# ================================================================
# STEP 5: CREATE COMPARISON TABLE
# ================================================================

results_df = pd.DataFrame(results)

model_order = [
    "DDPG",
    "TD3",
    "PPO",
    "SAC",
    "DE-TwinD3PG [Proposed]"
]

results_df["Methods"] = pd.Categorical(
    results_df["Methods"],
    categories=model_order,
    ordered=True
)

results_df = (
    results_df
    .sort_values("Methods")
    .reset_index(drop=True)
)

results_df["Methods"] = results_df["Methods"].astype(str)

print("\n" + "=" * 95)
print("COMPARATIVE PERFORMANCE EVALUATION")
print("=" * 95)

print(results_df.to_string(index=False))


COMPARATIVE PERFORMANCE EVALUATION
DE-TwinD3PG AGAINST RL BASELINES
       Methods          Accuracy  Precision  Recall  F1 Score
                  DDPG    0.88      0.86      0.84     0.85  
                   TD3    0.91      0.89      0.88     0.88  
                   PPO    0.90      0.87      0.89     0.88  
                   SAC    0.92      0.90      0.91     0.90  
DE-TwinD3PG [Proposed]    0.96      0.96      0.96     0.96  
